# Step 3: Calculate Celltype Proportions and Distance Analysis

This step visualizes celltype proportions across timepoints and computes distance analysis

## Setup and imports

In [ ]:
# Imports
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import squidpy as sq
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
import glob
import os
import shutil
import matplotlib.cm as cm
from scipy.stats import wilcoxon, zscore, gmean, pearsonr, norm, mannwhitneyu
from scipy import sparse
from statsmodels.stats.multitest import multipletests, fdrcorrection
from joblib import Parallel, delayed
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
from matplotlib.patches import Circle
import plotly.graph_objects as go
from plotly.colors import label_rgb
import math
import matplotlib.cm as mcm
import matplotlib.colors as mcolors

In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/path/to/integrated/processed_data/'
outdir = '/path/to/integrated/out/'
figdir = '/path/to/integrated/figures/'

In [ ]:
# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

In [ ]:
# Define color palette for each patient
patient_colors = {
    'Patient1': "#d9d9d9",
    'Patient4': "#bdbdbd",
    'Patient5': "#969696",
    'Patient2': "#636363",
    'Patient3': "#252525"
}

## Load data

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

## Plot spatial maps

In [ ]:
# Plot settings and color map
sample_key = "sample"
patient_key = "patient"
timepoint_key = "timepoint"
group_key = "celltype"
scalebar_len = 500  # microns

if "celltype_colors" in adata.uns and pd.api.types.is_categorical_dtype(adata.obs[group_key]):
    cats = adata.obs[group_key].cat.categories
    colors = adata.uns["celltype_colors"]
    color_map = {c: colors[i] for i, c in enumerate(cats)}
else:
    cats = sorted(pd.unique(adata.obs[group_key]))
    color_map = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}

In [ ]:
# Build metadata and spatial bounds
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()

mins, maxs = [], []
for s in meta[sample_key].unique():
    sub = adata[adata.obs[sample_key] == s]
    coords = sub.obsm["spatial"]
    mins.append(coords.min(axis=0))
    maxs.append(coords.max(axis=0))
mins = np.vstack(mins)
maxs = np.vstack(maxs)
global_min = mins.min(axis=0)
global_max = maxs.max(axis=0)

In [ ]:
# Plot
for _, row in meta.iterrows():
    sample = row[sample_key]
    patient = row[patient_key]
    tp = row[timepoint_key]

    sub = adata[adata.obs[sample_key] == sample]
    coords = sub.obsm["spatial"]
    labels = sub.obs[group_key].astype(str).values
    colors = [color_map.get(x, "gray") for x in labels]

    fig, ax = plt.subplots(figsize=(9, 9))  # larger size
    ax.scatter(coords[:, 0], coords[:, 1], s=2, c=colors, linewidths=0, alpha=0.8)

    ax.set_xlim(global_min[0], global_max[0])
    ax.set_ylim(global_min[1], global_max[1])
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(f"{patient} {tp}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    if str(tp).upper() == "DX":
        x0 = global_min[0] + 0.05 * (global_max[0] - global_min[0])
        y0 = global_max[1] - 0.05 * (global_max[1] - global_min[1])
        ax.plot([x0, x0 + scalebar_len], [y0, y0], color="black", linewidth=2)
        ax.text(x0, y0 - 0.02 * (global_max[1] - global_min[1]),
                f"{scalebar_len} µm", fontsize=8, ha="left", va="top")

    plt.tight_layout()
    plt.savefig(f"{figdir}spatial_{patient}_{tp}.png",
                format="png", bbox_inches="tight", transparent=True, dpi=600)
    plt.show()
    plt.close(fig)

## Plot neuroblasts and T and B cells only

In [ ]:
# Plot settings - subset to specific celltypes
sample_key = "sample"
patient_key = "patient"
timepoint_key = "timepoint"
group_key = "celltype"
scalebar_len = 500  # microns
celltypes_to_plot = ["Neuroblast", "T", "B"]

if "celltype_colors" in adata.uns and pd.api.types.is_categorical_dtype(adata.obs[group_key]):
    cats = adata.obs[group_key].cat.categories
    colors = adata.uns["celltype_colors"]
    color_map = {c: colors[i] for i, c in enumerate(cats)}
else:
    cats = sorted(pd.unique(adata.obs[group_key]))
    color_map = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}

color_map["T"] = "#000000"  # override T cell color to black
color_map["B"] = "#A2202C"  # override B cell color to maroon

# Build metadata and spatial bounds
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()

mins, maxs = [], []
for s in meta[sample_key].unique():
    sub = adata[adata.obs[sample_key] == s]
    coords = sub.obsm["spatial"]
    mins.append(coords.min(axis=0))
    maxs.append(coords.max(axis=0))
mins = np.vstack(mins)
maxs = np.vstack(maxs)
global_min = mins.min(axis=0)
global_max = maxs.max(axis=0)


# Plot - only Neuroblast, T, B cells (others shown as light gray background)
for _, row in meta.iterrows():
    sample = row[sample_key]
    patient = row[patient_key]
    tp = row[timepoint_key]

    sub = adata[adata.obs[sample_key] == sample]
    coords = sub.obsm["spatial"]
    labels = sub.obs[group_key].astype(str).values

    # Separate background and foreground cells
    bg_mask = ~np.isin(labels, celltypes_to_plot)
    fg_mask = np.isin(labels, celltypes_to_plot)

    fig, ax = plt.subplots(figsize=(9, 9))

    # Plot background cells first (light gray, low alpha)
    if bg_mask.any():
        ax.scatter(coords[bg_mask, 0], coords[bg_mask, 1],
                   s=2, c="lightgray", linewidths=0, alpha=0.2)

    # Plot foreground cells on top
    fg_colors = [color_map.get(x, "gray") for x in labels[fg_mask]]
    ax.scatter(coords[fg_mask, 0], coords[fg_mask, 1],
               s=2, c=fg_colors, linewidths=0, alpha=0.8)

    ax.set_xlim(global_min[0], global_max[0])
    ax.set_ylim(global_min[1], global_max[1])
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(f"{patient} {tp}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    if str(tp).upper() == "DX":
        x0 = global_min[0] + 0.05 * (global_max[0] - global_min[0])
        y0 = global_max[1] - 0.05 * (global_max[1] - global_min[1])
        ax.plot([x0, x0 + scalebar_len], [y0, y0], color="black", linewidth=2)
        ax.text(x0, y0 - 0.02 * (global_max[1] - global_min[1]),
                f"{scalebar_len} µm", fontsize=8, ha="left", va="top")

    plt.tight_layout()
    plt.savefig(f"{figdir}spatial_{patient}_{tp}_NTB.png",
                format="png", bbox_inches="tight", transparent=True, dpi=600)
    plt.show()
    plt.close(fig)

## Plot TLS

In [ ]:
sample_id = 'Patient1_DX' # sample with TLS
subset = adata[adata.obs['sample'] == sample_id].copy()

In [ ]:
# Plot CD4 and CXCL13
sc.pl.spatial(
    subset,
    color=['CD4', 'CXCL13'],
    spot_size=30,
    cmap='YlGnBu',
    show=False
)

plt.savefig(f"{figdir}tls_{sample_id}_genes.png",
            format="png", bbox_inches="tight", transparent=True, dpi=600)
plt.show()
plt.close()

In [ ]:
# Use celltypes_all (subtypes)
celltypes_all = subset.obs['celltypes_all'].cat.categories.tolist()

# Build parent-color map (celltypes_all > celltype > color)
df_map = (
    adata.obs[["celltypes_all", "celltype"]]
    .dropna()
    .drop_duplicates()
)

# Use existing celltype colors
main_types = list(adata.obs["celltype"].cat.categories)
main_colors = adata.uns["celltype_colors"]
celltype_color_map = {ct: main_colors[i] for i, ct in enumerate(main_types)}

# Map each celltypes_all to its parent celltype color
subtype_to_parent = dict(zip(df_map["celltypes_all"], df_map["celltype"]))
subtype_color_map = {
    st: celltype_color_map.get(subtype_to_parent.get(st, None), "lightgray")
    for st in celltypes_all
}

# Ensure palette matches order of celltypes_all categories
subset.obs["celltypes_all"] = subset.obs["celltypes_all"].astype("category")
ordered_subtypes = list(subset.obs["celltypes_all"].cat.categories)
subtype_colors = [subtype_color_map.get(st, "lightgray") for st in ordered_subtypes]

In [ ]:
# Plot one figure per celltypes_all subtype and save
for ct in celltypes_all:
    sc.pl.spatial(
    subset,
    color="celltypes_all",
    groups=None,
    palette=subtype_colors,
    spot_size=30,
    show=False,
    na_color="#000000"
    )

# overlay just the subtype with larger size
    sc.pl.spatial(
    subset,
    color="celltypes_all",
    groups=[ct],
    palette=subtype_colors,
    spot_size=50,
    show=False,
    na_color="#000000"
    )

    safe_ct = ct.replace("/", "_").replace(" ", "_")
    plt.savefig(f"{figdir}tls_{sample_id}_{safe_ct}.png", format="png", bbox_inches="tight", transparent=True, dpi=600)
    
    plt.show()
    plt.close()

## Distance analysis
For neighborhood enrichment, we keep the **spatial graph fixed** and **shuffle cell labels** to generate a null distribution of neighbor counts. This preserves the tissue geometry while testing whether the observed cell–cell adjacency is stronger or weaker than expected by chance. We use the permutation mean as the expected count and **empirical permutation p-values** for significance. This is appropriate here because spatial adjacency is constrained by physical coordinates, so label permutations on a fixed graph reflect the correct null for cell-type proximity.

### Helper functions for distance analyses

In [ ]:
FONT_SIZE = 5
FIGSIZE_IN = (7 / 2.54, 7 / 2.54)

In [ ]:
def _apply_heatmap_tick_style(ax):
    ax.tick_params(axis="both", which="both", width=0.5, length=2)


def _to_dense(x):
    if sparse.issparse(x):
        return x.A
    return np.asarray(x)


def _stars_from_q(q):
    if pd.isna(q):
        return ""
    if q < 1e-3:
        return "***"
    if q < 1e-2:
        return "**"
    if q < 5e-2:
        return "*"
    return ""


def _stars_from_pct(pct):
    if pd.isna(pct):
        return ""
    if pct >= 75:
        return "***"
    if pct >= 50:
        return "**"
    if pct >= 30:
        return "*"
    return ""


def sign_consistency(x):
    return (x > 0).all() or (x < 0).all()


def get_common_categories(obs_series):
    if pd.api.types.is_categorical_dtype(obs_series):
        return list(obs_series.cat.categories)
    return sorted(pd.unique(obs_series))


def build_subtype_order_from_parent(
    adata,
    parent_order,
    subtype_key="celltypes_all",
    parent_key="celltype",
):
    df_map = (
        adata.obs[[subtype_key, parent_key]]
        .dropna()
        .drop_duplicates()
    )
    subtype_order = []
    for parent in parent_order:
        subtypes = (
            df_map.loc[df_map[parent_key] == parent, subtype_key]
            .unique()
            .tolist()
        )
        subtype_order.extend(sorted(subtypes))
    return subtype_order


def _interaction_count_matrix_from_adj(adj_csr, codes, n_cls):
    """
    Compute interaction counts (source type -> neighbor type) from CSR graph.
    Returns (n_cls, n_cls) array.
    """
    if not sparse.isspmatrix_csr(adj_csr):
        adj_csr = adj_csr.tocsr()

    indptr = adj_csr.indptr
    indices = adj_csr.indices
    counts = np.zeros((n_cls, n_cls), dtype=np.float64)

    for i in range(adj_csr.shape[0]):
        src = codes[i]
        start, end = indptr[i], indptr[i + 1]
        nbr_idx = indices[start:end]
        if nbr_idx.size == 0:
            continue
        nbr_labels = codes[nbr_idx]
        counts[src, :] += np.bincount(nbr_labels, minlength=n_cls).astype(np.float64)

    return counts


def _cluster_order_from_log2(
    df_log2,
    use_symmetric_for_order=True,
    cluster_method="average",
    cluster_metric="euclidean",
):
    ordered = list(df_log2.index)
    Z = None

    cluster_mat = df_log2.copy()
    if use_symmetric_for_order:
        cluster_mat = (cluster_mat + cluster_mat.T) / 2.0

    X = cluster_mat.to_numpy(dtype=float)
    row_means = np.nanmean(X, axis=1)
    row_means[np.isnan(row_means)] = 0.0
    nan_r, nan_c = np.where(np.isnan(X))
    X[nan_r, nan_c] = row_means[nan_r]
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if X.shape[0] >= 2:
        d = pdist(X, metric=cluster_metric)
        if not np.allclose(d, 0):
            Z = linkage(d, method=cluster_method)
            leaves = dendrogram(Z, no_plot=True)["leaves"]
            ordered = [df_log2.index[i] for i in leaves]

    return ordered, Z


def get_consistent_pairs(df_pairs, focus_labels=None):
    """
    Returns DataFrame with consistent-sign pairs and direction.
    """
    sig = df_pairs.query("significant and not low_expected").copy()
    if focus_labels is not None:
        focus_labels = set(focus_labels)
        sig = sig[
            sig["source"].isin(focus_labels) | sig["neighbor"].isin(focus_labels)
        ].copy()

    consistency = sig.groupby(["source", "neighbor"])["log2_oe"].agg(
        n_sig="size",
        all_same_sign=sign_consistency,
        direction=lambda v: "enriched" if (v > 0).all() else ("depleted" if (v < 0).all() else "mixed"),
    ).reset_index()

    return consistency[consistency["all_same_sign"]].copy()


def plot_nhood(
    adata_sub,
    group_key="celltype",
    sample_label="",
    n_perms=2000,
    seed=0,
    alpha=0.05,
    pseudocount=1.0,
    clip=(-2, 2),
    mask_upper_triangle=True,
    coord_type="generic",
    spatial_key="spatial",
    neigh_kwargs=None,
    figsize_left=FIGSIZE_IN,
    figsize_right=FIGSIZE_IN,
    value_fmt="{:.2f}",
    annot_fontsize=FONT_SIZE,
    left_text_scale=1.0,
    connectivities_key="spatial_connectivities",
    n_jobs=8,
    figdir=None,
    outdir=None,
    label_order=None,
    filename_prefix="",
    cluster=True,
    show_dendrogram=True,
    use_symmetric_for_order=True,
    cluster_method="average",
    cluster_metric="euclidean",
):
    if neigh_kwargs is None:
        neigh_kwargs = {}

    # 1) Build spatial graph + neighborhood enrichment
    sq.gr.spatial_neighbors(
        adata_sub,
        coord_type=coord_type,
        spatial_key=spatial_key,
        **neigh_kwargs
    )
    sq.gr.nhood_enrichment(
        adata_sub,
        cluster_key=group_key,
        n_perms=n_perms,
        seed=seed,
        copy=False
    )

    key = f"{group_key}_nhood_enrichment"
    obs = _to_dense(adata_sub.uns[key]["count"]).astype(float)

    cats = adata_sub.obs[group_key]
    if not pd.api.types.is_categorical_dtype(cats):
        cats = cats.astype("category")
        adata_sub.obs[group_key] = cats
    labels = list(cats.cat.categories)
    n_cls = len(labels)

    if connectivities_key not in adata_sub.obsp:
        raise KeyError(
            f"'{connectivities_key}' not found in adata_sub.obsp. "
            f"Available keys: {list(adata_sub.obsp.keys())}"
        )

    adj = adata_sub.obsp[connectivities_key]
    if not sparse.isspmatrix_csr(adj):
        adj = adj.tocsr()

    base_codes = cats.cat.codes.to_numpy()
    if np.any(base_codes < 0):
        raise ValueError(f"{group_key} contains NaN/unassigned categories; drop or fill first.")

    # 2) Permutation expected counts + empirical p-values
    seeds = seed + np.arange(n_perms)

    def _perm_chunk(seed_chunk):
        exp_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        exp_sq_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        ge_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        le_local = np.zeros((n_cls, n_cls), dtype=np.float64)

        for s in seed_chunk:
            perm_codes = base_codes.copy()
            rng_local = np.random.default_rng(s)
            rng_local.shuffle(perm_codes)
            perm_counts = _interaction_count_matrix_from_adj(adj, perm_codes, n_cls)

            exp_local += perm_counts
            exp_sq_local += perm_counts ** 2
            ge_local += (perm_counts >= obs)
            le_local += (perm_counts <= obs)

        return exp_local, exp_sq_local, ge_local, le_local

    if n_jobs is None or n_jobs <= 1:
        exp_sum, exp_sq_sum, ge_sum, le_sum = _perm_chunk(seeds)
    else:
        n_jobs_eff = min(int(n_jobs), n_perms)
        seed_chunks = np.array_split(seeds, n_jobs_eff)
        parts = Parallel(n_jobs=n_jobs_eff, backend="loky")(
            delayed(_perm_chunk)(chunk) for chunk in seed_chunks
        )
        exp_sum = np.sum([p[0] for p in parts], axis=0)
        exp_sq_sum = np.sum([p[1] for p in parts], axis=0)
        ge_sum = np.sum([p[2] for p in parts], axis=0)
        le_sum = np.sum([p[3] for p in parts], axis=0)

    exp_perm = exp_sum / float(n_perms)
    exp_var = np.clip((exp_sq_sum / float(n_perms)) - (exp_perm ** 2), 0, None)
    exp_sd = np.sqrt(exp_var)
    z_perm = (obs - exp_perm) / (exp_sd + 1e-9)
    log2_oe = np.log2((obs + pseudocount) / (exp_perm + pseudocount))

    p_upper = (ge_sum + 1.0) / (n_perms + 1.0)
    p_lower = (le_sum + 1.0) / (n_perms + 1.0)
    p = np.clip(2.0 * np.minimum(p_upper, p_lower), 0, 1)

    reject_flat, q_flat = fdrcorrection(p.ravel(), alpha=alpha)
    significant = reject_flat.reshape(p.shape)
    q = q_flat.reshape(p.shape)

    # 3) Tidy output (statistics)
    df_pairs = pd.DataFrame({
        "source": np.repeat(labels, len(labels)),
        "neighbor": labels * len(labels),
        "obs": obs.ravel(),
        "exp_perm": exp_perm.ravel(),
        "log2_oe": log2_oe.ravel(),
        "z": z_perm.ravel(),
        "p": p.ravel(),
        "q": q.ravel(),
        "significant": significant.ravel(),
    })
    df_pairs["low_expected"] = df_pairs["exp_perm"] < 5

    # 4) Build matrices
    df_log2 = pd.DataFrame(log2_oe, index=labels, columns=labels)
    df_obs = pd.DataFrame(obs, index=labels, columns=labels)
    df_q = pd.DataFrame(q, index=labels, columns=labels)
    df_lowexp = pd.DataFrame(exp_perm < 5, index=labels, columns=labels)

    # Optional manual order first
    if label_order is not None:
        ordered_manual = [x for x in label_order if x in labels]
        leftovers = [x for x in labels if x not in ordered_manual]
        plot_order = ordered_manual + leftovers
        df_log2 = df_log2.reindex(index=plot_order, columns=plot_order)
        df_obs = df_obs.reindex(index=plot_order, columns=plot_order)
        df_q = df_q.reindex(index=plot_order, columns=plot_order)
        df_lowexp = df_lowexp.reindex(index=plot_order, columns=plot_order)

    # Optional clustering order (like plot_consensus_heatmaps)
    Z = None
    if cluster:
        ordered, Z = _cluster_order_from_log2(
            df_log2,
            use_symmetric_for_order=use_symmetric_for_order,
            cluster_method=cluster_method,
            cluster_metric=cluster_metric,
        )
        df_log2 = df_log2.reindex(index=ordered, columns=ordered)
        df_obs = df_obs.reindex(index=ordered, columns=ordered)
        df_q = df_q.reindex(index=ordered, columns=ordered)
        df_lowexp = df_lowexp.reindex(index=ordered, columns=ordered)

    ann = df_log2.applymap(lambda v: value_fmt.format(v))
    stars = df_q.applymap(_stars_from_q).mask(df_lowexp, "")
    ann = ann + "\n" + stars

    vmin, vmax = clip
    df_log2_clipped = df_log2.clip(lower=vmin, upper=vmax)

    mask_tri = None
    if mask_upper_triangle:
        mask_tri = np.triu(np.ones(df_log2_clipped.shape, dtype=bool), k=1)

    safe_label = sample_label.replace(" ", "_")
    base_name = f"{filename_prefix}nhood_enrichment_{safe_label}"
    fig_base_dir = figdir if figdir is not None else outdir

    # Save values/statistics CSVs for each heatmap
    if outdir is not None:
        # Effect heatmap
        df_log2.to_csv(f"{outdir}{base_name}_effect_values.csv")
        df_pairs.to_csv(f"{outdir}{base_name}_effect_stats.csv", index=False)

        # Count heatmap
        df_obs.to_csv(f"{outdir}{base_name}_counts_values_raw.csv")
        np.log10(df_obs + 1.0).to_csv(f"{outdir}{base_name}_counts_values_log10.csv")
        df_pairs.to_csv(f"{outdir}{base_name}_counts_stats.csv", index=False)

    # Left panel: effect size (+ optional dendrogram)
    if cluster and show_dendrogram and Z is not None:
        fig_left = plt.figure(figsize=figsize_left)
        gs = fig_left.add_gridspec(2, 1, height_ratios=[0.24, 1.5], hspace=0.2)

        ax_den = fig_left.add_subplot(gs[0, 0])
        dendrogram(
            Z,
            labels=list(df_log2.index),
            no_labels=True,
            color_threshold=0,
            above_threshold_color="black",
            link_color_func=lambda k: "black",
            ax=ax_den
        )
        for coll in ax_den.collections:
            coll.set_linewidth(0.5)
        ax_den.set_xticks([])
        ax_den.set_yticks([])
        for spine in ax_den.spines.values():
            spine.set_visible(False)
        ax_den.set_title("Hierarchical clustering", fontsize=FONT_SIZE)

        ax0 = fig_left.add_subplot(gs[1, 0])
    else:
        fig_left, ax0 = plt.subplots(figsize=figsize_left)

    mask_color = df_lowexp.copy()
    if mask_tri is not None:
        mask_color = mask_color | mask_tri

    sns.heatmap(
        df_log2_clipped,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        square=True,
        xticklabels=df_log2.columns,
        yticklabels=df_log2.index,
        mask=mask_color,
        annot=False,
        linewidths=0.3,
        linecolor="lightgray",
        cbar_kws={"label": "log2(Observed / Expected_perm_mean) (color clipped)"},
        ax=ax0
    )

    mask_white = ~(df_lowexp & ~(mask_tri if mask_tri is not None else False))
    sns.heatmap(
        df_log2_clipped,
        cmap=["white"],
        square=True,
        mask=mask_white,
        cbar=False,
        linewidths=0.3,
        linecolor="lightgray",
        xticklabels=True,
        yticklabels=True,
        ax=ax0
    )

    left_annot_fs = FONT_SIZE
    for i in range(df_log2.shape[0]):
        for j in range(df_log2.shape[1]):
            if mask_tri is not None and mask_tri[i, j]:
                continue
            val = df_log2.iloc[i, j]
            if pd.isna(val):
                continue
            text_color = "black" if df_lowexp.iloc[i, j] else ("white" if abs(val) > 1.5 else "black")
            ax0.text(
                j + 0.5, i + 0.5, ann.iloc[i, j],
                ha="center", va="center",
                fontsize=left_annot_fs, color=text_color
            )

    ax0.set_title(
        f"log2(Obs/Exp_perm_mean) with FDR stars (q<{alpha})" + (f"\n{sample_label}" if sample_label else ""),
        fontsize=FONT_SIZE
    )
    ax0.set_xlabel("Neighbor cell type", fontsize=FONT_SIZE)
    ax0.set_ylabel("Source cell type", fontsize=FONT_SIZE)
    ax0.set_xticklabels(ax0.get_xticklabels(), rotation=90, ha="center", fontsize=FONT_SIZE)
    ax0.set_yticklabels(ax0.get_yticklabels(), rotation=0, va="center", fontsize=FONT_SIZE)
    _apply_heatmap_tick_style(ax0)

    plt.tight_layout()
    if fig_base_dir is not None:
        plt.savefig(
            f"{fig_base_dir}{base_name}_effect.pdf",
            format="pdf", bbox_inches="tight", transparent=True
        )
    plt.show()
    plt.close(fig_left)

    # Right panel: observed counts
    fig_right, ax1 = plt.subplots(figsize=figsize_right)
    sns.heatmap(
        np.log10(df_obs + 1.0),
        cmap="viridis",
        square=True,
        xticklabels=df_obs.columns,
        yticklabels=df_obs.index,
        mask=mask_tri,
        linewidths=0.3,
        linecolor="lightgray",
        cbar_kws={"label": "log10(Observed count + 1)"},
        ax=ax1
    )
    ax1.set_title("Observed neighbor counts" + (f"\n{sample_label}" if sample_label else ""), fontsize=FONT_SIZE)
    ax1.set_xlabel("Neighbor cell type", fontsize=FONT_SIZE)
    ax1.set_ylabel("Source cell type", fontsize=FONT_SIZE)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90, ha="center", fontsize=FONT_SIZE)
    ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, va="center", fontsize=FONT_SIZE)
    _apply_heatmap_tick_style(ax1)

    plt.tight_layout()
    if fig_base_dir is not None:
        plt.savefig(
            f"{fig_base_dir}{base_name}_counts.pdf",
            format="pdf", bbox_inches="tight", transparent=True
        )
    plt.show()
    plt.close(fig_right)

    return df_pairs


def build_consensus_table(
    df_pairs,
    min_samples_sig=0.5,
    include_dispersion=True,
    out_csv=None,
):
    """
    Build consensus summary from per-sample df_pairs.
    Uses sample-level reproducibility:
      sig_valid = significant AND not low_expected
      pct_sig = percentage of samples with sig_valid
    """
    if df_pairs.empty:
        raise ValueError("df_pairs is empty.")

    x = df_pairs.copy()
    x["sig_valid"] = (x["significant"] & ~x["low_expected"]).astype(float)

    agg = {
        "mean_log2_oe": ("log2_oe", "mean"),
        "mean_obs": ("obs", "mean"),
        "pct_sig": ("sig_valid", "mean"),
    }
    if include_dispersion:
        agg.update({
            "sd_log2_oe": ("log2_oe", "std"),
            "iqr_log2_oe": ("log2_oe", lambda v: np.nanpercentile(v, 75) - np.nanpercentile(v, 25)),
        })

    consensus = x.groupby(["source", "neighbor"]).agg(**agg).reset_index()
    consensus["pct_sig"] = 100 * consensus["pct_sig"]
    consensus["consensus_sig"] = consensus["pct_sig"] >= (min_samples_sig * 100)

    if out_csv is not None:
        consensus.to_csv(out_csv, index=False)

    return consensus


def plot_consensus_heatmaps(
    consensus,
    categories,
    min_samples_sig=0.5,
    pair_set=None,
    clip=(-2, 2),
    box_text_size=FONT_SIZE,
    title_size=FONT_SIZE,
    axis_label_size=FONT_SIZE,
    tick_label_size=FONT_SIZE,
    figsize_left=FIGSIZE_IN,
    figsize_right=FIGSIZE_IN,
    cluster=True,
    show_dendrogram=False,
    use_symmetric_for_order=True,
    cluster_method="average",
    cluster_metric="euclidean",
    left_title="Consensus Enrichment",
    right_title="Mean Observed Neighbor Counts",
    figdir=None,
    outdir=None,
    filename_prefix="",
):
    """
    Plot consensus heatmaps with optional clustering + dendrogram.
    Saves per-heatmap value CSV + stats CSV into outdir.
    Saves figures as PDF.
    """
    if consensus.empty:
        raise ValueError("consensus is empty.")

    cats = list(categories)

    df_log2 = consensus.pivot(index="source", columns="neighbor", values="mean_log2_oe").reindex(index=cats, columns=cats)
    df_obs = consensus.pivot(index="source", columns="neighbor", values="mean_obs").reindex(index=cats, columns=cats)
    df_pct = consensus.pivot(index="source", columns="neighbor", values="pct_sig").reindex(index=cats, columns=cats)

    sig_mask = df_pct >= (min_samples_sig * 100)

    # ordering
    Z = None
    ordered = cats
    if cluster:
        ordered, Z = _cluster_order_from_log2(
            df_log2,
            use_symmetric_for_order=use_symmetric_for_order,
            cluster_method=cluster_method,
            cluster_metric=cluster_metric,
        )

    df_log2 = df_log2.reindex(index=ordered, columns=ordered)
    df_obs = df_obs.reindex(index=ordered, columns=ordered)
    df_pct = df_pct.reindex(index=ordered, columns=ordered)
    sig_mask = sig_mask.reindex(index=ordered, columns=ordered)

    vmin, vmax = clip
    fig_base_dir = figdir if figdir is not None else outdir
    base_name = f"{filename_prefix}consensus"

    # Save values/statistics CSVs for each heatmap
    if outdir is not None:
        # Left heatmap
        df_log2.to_csv(f"{outdir}{base_name}_log2oe_values.csv")
        left_stats = consensus.copy()
        left_stats["min_samples_sig_threshold_pct"] = min_samples_sig * 100
        left_stats.to_csv(f"{outdir}{base_name}_log2oe_stats.csv", index=False)

        # Right heatmap
        df_obs.to_csv(f"{outdir}{base_name}_mean_obs_values_raw.csv")
        np.log10(df_obs + 1).to_csv(f"{outdir}{base_name}_mean_obs_values_log10.csv")
        right_stats = consensus.copy()
        right_stats.to_csv(f"{outdir}{base_name}_mean_obs_stats.csv", index=False)

    # Left figure
    if cluster and show_dendrogram and Z is not None:
        fig_left = plt.figure(figsize=figsize_left)
        gs = fig_left.add_gridspec(2, 1, height_ratios=[0.24, 1.5], hspace=0.2)

        ax_den = fig_left.add_subplot(gs[0, 0])
        dendrogram(
            Z,
            labels=ordered,
            no_labels=True,
            color_threshold=0,
            above_threshold_color="black",
            link_color_func=lambda k: "black",
            ax=ax_den
        )
        for coll in ax_den.collections:
            coll.set_linewidth(0.5)
        ax_den.set_xticks([])
        ax_den.set_yticks([])
        for spine in ax_den.spines.values():
            spine.set_visible(False)
        #ax_den.set_title("Hierarchical clustering", fontsize=FONT_SIZE)
        ax_den.set_title("Hierarchical clustering", fontsize=title_size)  # changed

        ax0 = fig_left.add_subplot(gs[1, 0])
    else:
        fig_left, ax0 = plt.subplots(figsize=figsize_left)

    sns.heatmap(
        df_log2.clip(vmin, vmax),
        cmap="RdBu_r",
        center=0,
        vmin=vmin, vmax=vmax,
        xticklabels=df_log2.columns,
        yticklabels=df_log2.index,
        square=True,
        linewidths=0.3,
        linecolor="lightgray",
        mask=~sig_mask,
        cbar_kws={"label": "Mean log2(Obs/Exp_perm_mean)"},
        ax=ax0,
        annot=False
    )

    ax0.set_xticks(np.arange(len(df_log2.columns)) + 0.5)
    ax0.set_yticks(np.arange(len(df_log2.index)) + 0.5)
    #ax0.set_xticklabels(df_log2.columns, rotation=90, ha="center", fontsize=FONT_SIZE)
    #ax0.set_yticklabels(df_log2.index, rotation=0, va="center", fontsize=FONT_SIZE)
    ax0.set_xticklabels(df_log2.columns, rotation=90, ha="center", fontsize=tick_label_size)  # changed
    ax0.set_yticklabels(df_log2.index, rotation=0, va="center", fontsize=tick_label_size)      # changed
    ax0.tick_params(axis="x", bottom=True, labelbottom=True)
    ax0.tick_params(axis="y", left=True, labelleft=True)

    sns.heatmap(
        df_log2.clip(vmin, vmax),
        cmap=["white"],
        square=True,
        linewidths=0.3,
        linecolor="lightgray",
        mask=sig_mask,
        cbar=False,
        xticklabels=True,
        yticklabels=True,
        ax=ax0
    )

    ax0.set_xticks(np.arange(len(df_log2.columns)) + 0.5)
    ax0.set_yticks(np.arange(len(df_log2.index)) + 0.5)
    #ax0.set_xticklabels(df_log2.columns, rotation=90, ha="center", fontsize=FONT_SIZE)
    #ax0.set_yticklabels(df_log2.index, rotation=0, va="center", fontsize=FONT_SIZE)
    ax0.set_xticklabels(df_log2.columns, rotation=90, ha="center", fontsize=tick_label_size)  # changed
    ax0.set_yticklabels(df_log2.index, rotation=0, va="center", fontsize=tick_label_size)      # changed
    ax0.tick_params(axis="x", bottom=True, labelbottom=True)
    ax0.tick_params(axis="y", left=True, labelleft=True)

    for i, s in enumerate(df_log2.index):
        for j, n in enumerate(df_log2.columns):
            val = df_log2.iloc[i, j]
            if pd.isna(val):
                continue
            is_sig = bool(sig_mask.iloc[i, j])
            star = _stars_from_pct(df_pct.iloc[i, j]) if is_sig else ""
            txt = f"{val:.2f}" + (f"\n{star}" if star else "")
            text_color = "black" if not is_sig else ("white" if abs(val) > 1.5 else "black")
            ax0.text(j + 0.5, i + 0.5, txt, ha="center", va="center", fontsize=box_text_size, color=text_color)

    if pair_set is not None:
        for i, s in enumerate(df_log2.index):
            for j, n in enumerate(df_log2.columns):
                if (s, n) not in pair_set:
                    continue
                val = df_log2.iloc[i, j]
                if pd.isna(val):
                    continue
                is_sig = bool(sig_mask.iloc[i, j])
                edge_color = "black" if not is_sig else ("white" if abs(val) > 1.5 else "black")
                circ = Circle((j + 0.5, i + 0.5), radius=0.45, edgecolor=edge_color, facecolor="none", linewidth=0.3, zorder=10)
                ax0.add_patch(circ)

    #ax0.set_title(left_title, fontsize=FONT_SIZE)
    #ax0.set_xlabel("Neighbor cell type", fontsize=FONT_SIZE)
    #ax0.set_ylabel("Source cell type", fontsize=FONT_SIZE)
    #ax0.set_xticklabels(ax0.get_xticklabels(), rotation=90, ha="center")
    #ax0.set_yticklabels(ax0.get_yticklabels(), rotation=0, va="center")
    ax0.set_title(left_title, fontsize=title_size)                         # changed
    ax0.set_xlabel("Neighbor cell type", fontsize=axis_label_size)         # changed
    ax0.set_ylabel("Source cell type", fontsize=axis_label_size)           # changed
    ax0.set_xticklabels(ax0.get_xticklabels(), rotation=90, ha="center", fontsize=tick_label_size)  # changed
    ax0.set_yticklabels(ax0.get_yticklabels(), rotation=0, va="center", fontsize=tick_label_size)    # changed
    
    _apply_heatmap_tick_style(ax0)

    plt.tight_layout()
    if fig_base_dir is not None:
        plt.savefig(
            f"{fig_base_dir}{base_name}_log2oe.pdf",
            format="pdf", bbox_inches="tight", transparent=True
        )
    plt.show()
    plt.close(fig_left)

    # Right figure
    fig_right, ax1 = plt.subplots(figsize=figsize_right)
    sns.heatmap(
        np.log10(df_obs + 1),
        cmap="viridis",
        square=True,
        xticklabels=df_log2.columns,
        yticklabels=df_log2.index,
        linewidths=0.3,
        linecolor="lightgray",
        cbar_kws={"label": "log10(Mean Obs + 1)"},
        ax=ax1
    )
    ax1.set_title(right_title, fontsize=FONT_SIZE)
    ax1.set_xlabel("Neighbor cell type", fontsize=FONT_SIZE)
    ax1.set_ylabel("Source cell type", fontsize=FONT_SIZE)
    #ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90, ha="center")
    #ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, va="center")
    _apply_heatmap_tick_style(ax1)

    ax1.set_xticks(np.arange(len(df_log2.columns)) + 0.5)
    ax1.set_yticks(np.arange(len(df_log2.index)) + 0.5)
    ax1.set_xticklabels(df_log2.columns, rotation=90, ha="center", fontsize=FONT_SIZE)
    ax1.set_yticklabels(df_log2.index, rotation=0, va="center", fontsize=FONT_SIZE)
    ax1.tick_params(axis="x", bottom=True, labelbottom=True)
    ax1.tick_params(axis="y", left=True, labelleft=True)

    plt.tight_layout()
    if fig_base_dir is not None:
        plt.savefig(
            f"{fig_base_dir}{base_name}_mean_obs.pdf",
            format="pdf", bbox_inches="tight", transparent=True
        )
    plt.show()
    plt.close(fig_right)

    return {
        "ordered_categories": ordered,
        "df_log2": df_log2,
        "df_obs": df_obs,
        "df_pct": df_pct,
        "sig_mask": sig_mask,
    }

### Per-sample (main celltypes)

In [ ]:
# Run for all samples (main celltypes)
celltype_order = ["Neuroblast", "B", "T", "Schwann", "Endothelial", "Fibroblast", "Macrophage"]

all_df = []
samples = adata.obs["sample"].unique().tolist()

for sample in samples:
    print(f"Processing {sample}...")
    adata_sub = adata[adata.obs["sample"] == sample].copy()
    if adata_sub.n_obs < 100:
        continue

    df_pairs = plot_nhood(
        adata_sub,
        group_key="celltype",
        sample_label=f"{sample} (n={adata_sub.n_obs})",
        n_perms=5000,
        seed=0,
        alpha=0.05,
        pseudocount=1.0,
        clip=(-2, 2),
        mask_upper_triangle=False,
        coord_type="generic",
        spatial_key="spatial",
        value_fmt="{:.2f}",
        n_jobs=8,
        figdir=outdir,
        outdir=outdir,
        filename_prefix="celltype_",
        label_order=celltype_order,   # <- enforce this order
        cluster=False,                # <- keep manual order
        show_dendrogram=False
    )

    df_pairs["sample"] = sample
    all_df.append(df_pairs)

    display(
        df_pairs.query("significant and not low_expected")
                .sort_values("log2_oe", ascending=False)
                .head(20)
    )

# Save combined CSV
df_pairs_all = pd.concat(all_df, ignore_index=True) if len(all_df) else pd.DataFrame()
if not df_pairs_all.empty and outdir is not None:
    df_pairs_all.to_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv", index=False)

### Delta (main celltypes)

In [ ]:
df_pairs_all = pd.read_csv('/Users/yyj/Doc/1_dod_dec25/out_dist_ind/celltype_nhood_enrichment_all_samples.csv')

In [ ]:
celltype_order = ["Neuroblast", "B", "T", "Schwann", "Endothelial", "Fibroblast", "Macrophage"]

In [ ]:
timepoint_key = "timepoint"
sample_key = "sample"
patient_key = "patient"

if df_pairs_all.empty:
    raise ValueError("df_pairs_all is empty. Run per-sample analysis first.")

# Style (aligned with helper cell)
value_fmt = "{:.2f}"
vmin, vmax = -2, 2
figsize_delta = FIGSIZE_IN
title_size = FONT_SIZE
axis_label_size = FONT_SIZE
annot_fontsize = FONT_SIZE
tick_label_size = FONT_SIZE

# consistent ordering
present = set(df_pairs_all["source"]).union(set(df_pairs_all["neighbor"]))
common_categories = [c for c in celltype_order if c in present]

# attach patient + timepoint to each sample
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()
df_pairs_meta = df_pairs_all.merge(meta, on=sample_key, how="left")

# keep only DX/PT
df_pairs_meta = df_pairs_meta[df_pairs_meta[timepoint_key].isin(["DX", "PT"])].copy()

# per-patient heatmaps
for patient in sorted(df_pairs_meta[patient_key].dropna().unique()):
    df_p = df_pairs_meta[df_pairs_meta[patient_key] == patient].copy()

    dx = df_p[df_p[timepoint_key] == "DX"].set_index(["source", "neighbor"])
    pt = df_p[df_p[timepoint_key] == "PT"].set_index(["source", "neighbor"])

    # only keep pairs present in both
    paired = pt[["log2_oe"]].rename(columns={"log2_oe": "pt"}).join(
        dx[["log2_oe"]].rename(columns={"log2_oe": "dx"}),
        how="inner"
    )
    paired["delta"] = paired["pt"] - paired["dx"]
    delta = paired["delta"].reset_index()

    # matrix
    df_delta = delta.pivot(index="source", columns="neighbor", values="delta").reindex(
        index=common_categories, columns=common_categories
    )
    df_delta_clipped = df_delta.clip(lower=vmin, upper=vmax)

    # optional outputs: values + stats CSV per heatmap
    if outdir is not None:
        df_delta.to_csv(f"{outdir}nhood_enrichment_delta_PT_minus_DX_{patient}_values.csv")
        delta.to_csv(f"{outdir}nhood_enrichment_delta_PT_minus_DX_{patient}_stats.csv", index=False)

    # plot
    fig, ax = plt.subplots(figsize=figsize_delta)
    sns.heatmap(
        df_delta_clipped,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        square=True,
        xticklabels=common_categories,
        yticklabels=common_categories,
        annot=False,
        linewidths=0.3,
        linecolor="lightgray",
        cbar_kws={"label": "Δ log2(Obs/Exp_perm_mean) (PT − DX)"},
        ax=ax
    )

    # manual text coloring
    for i, s in enumerate(df_delta.index):
        for j, n in enumerate(df_delta.columns):
            val = df_delta.iloc[i, j]
            if pd.isna(val):
                continue
            text_color = "white" if abs(val) > 1.5 else "black"
            ax.text(
                j + 0.5, i + 0.5, value_fmt.format(val),
                ha="center", va="center",
                fontsize=annot_fontsize, color=text_color
            )

    ax.set_title(f"PT - DX Delta\n(Patient {patient})", fontsize=title_size)
    ax.set_xlabel("Neighbor cell type", fontsize=axis_label_size)
    ax.set_ylabel("Source cell type", fontsize=axis_label_size)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center", fontsize=tick_label_size)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, va="center", fontsize=tick_label_size)
    _apply_heatmap_tick_style(ax)

    plt.tight_layout()
    save_dir = figdir if figdir is not None else outdir
    if save_dir is not None:
        plt.savefig(
            f"{save_dir}nhood_enrichment_delta_PT_minus_DX_{patient}.pdf",
            format="pdf",
            bbox_inches="tight",
            transparent=True
        )
    plt.show()
    plt.close(fig)

In [ ]:
# Calculate delta stats
x = df_pairs_all.copy()
x["sample"] = x["sample"].astype(str).str.strip()

# parse patient/timepoint from sample names like Patient1_DX / Patient5_PT
parts = x["sample"].str.rsplit("_", n=1, expand=True)
x["patient"] = parts[0].str.strip()
x["timepoint"] = parts[1].str.strip().str.upper()

# keep only DX/PT
x = x[x["timepoint"].isin(["DX", "PT"])].copy()

# ----------------------------
# 2) Keep interactions of interest
# ----------------------------
targets = [
    ("B", "B"),
    ("B", "T"),
    ("T", "B"),
    ("T", "T"),
    ("Macrophage", "T"),
    ("T", "Macrophage"),
]

target_df = pd.DataFrame(targets, columns=["source", "neighbor"])
x = x.merge(target_df, on=["source", "neighbor"], how="inner")

# ----------------------------
# 3) Compute per-patient delta = PT - DX
# ----------------------------
wide = (
    x.pivot_table(
        index=["patient", "source", "neighbor"],
        columns="timepoint",
        values="log2_oe",
        aggfunc="mean"   # safe if any duplicates
    )
    .reset_index()
)

# require both DX and PT
wide = wide.dropna(subset=["DX", "PT"]).copy()
wide["delta_PT_minus_DX"] = wide["PT"] - wide["DX"]

# ----------------------------
# 4) Assign trajectory groups
# ----------------------------
rebuilding_patients = ["Patient1", "Patient2", "Patient3"]
desert_patients = ["Patient4", "Patient5"]

wide["trajectory_group"] = np.where(
    wide["patient"].isin(rebuilding_patients), "immune_rebuilding",
    np.where(wide["patient"].isin(desert_patients), "immune_desertification", np.nan)
)
wide = wide.dropna(subset=["trajectory_group"]).copy()

# save per-patient deltas
wide.to_csv(f"{outdir}fig3c_selected_interaction_deltas_by_patient.csv", index=False)

# ----------------------------
# 5) Rank-sum tests per interaction
#    One-sided hypothesis: rebuilding has LOWER delta than desertification
# ----------------------------
rows = []
for s, n in targets:
    d = wide[(wide["source"] == s) & (wide["neighbor"] == n)].copy()

    rb = d.loc[d["trajectory_group"] == "immune_rebuilding", "delta_PT_minus_DX"].values
    ds = d.loc[d["trajectory_group"] == "immune_desertification", "delta_PT_minus_DX"].values

    if len(rb) == 0 or len(ds) == 0:
        rows.append({
            "source": s, "neighbor": n,
            "n_rebuilding": len(rb), "n_desertification": len(ds),
            "U_stat": np.nan, "p_one_sided_less": np.nan, "p_two_sided": np.nan,
            "mean_delta_rebuilding": np.nan, "mean_delta_desertification": np.nan
        })
        continue

    U_less, p_less = mannwhitneyu(rb, ds, alternative="less")
    U_2s, p_2s = mannwhitneyu(rb, ds, alternative="two-sided")

    rows.append({
        "source": s,
        "neighbor": n,
        "n_rebuilding": int(len(rb)),
        "n_desertification": int(len(ds)),
        "mean_delta_rebuilding": float(np.mean(rb)),
        "mean_delta_desertification": float(np.mean(ds)),
        "median_delta_rebuilding": float(np.median(rb)),
        "median_delta_desertification": float(np.median(ds)),
        "U_stat": float(U_less),
        "p_one_sided_less": float(p_less),
        "p_two_sided": float(p_2s),
        "min_possible_one_sided_p_with_n3_vs_n2": 0.1
    })

stats_df = pd.DataFrame(rows)

# BH-FDR on one-sided p-values across the 6 interactions
valid = stats_df["p_one_sided_less"].notna()
if valid.any():
    _, qvals, _, _ = multipletests(stats_df.loc[valid, "p_one_sided_less"], method="fdr_bh")
    stats_df.loc[valid, "p_one_sided_less_fdr_bh"] = qvals

# save stats
stats_df.to_csv(f"{outdir}fig3c_selected_interaction_mannwhitney_stats.csv", index=False)

display(wide)
display(stats_df)

### Dendogram consensus (main celltypes)
Optional cluster=True and show_dendrogram=False in plot_consensus_heatmaps function

In [ ]:
# Consistency of sign among significant samples (T/B interactions only)

if df_pairs_all.empty:
    raise ValueError("df_pairs_all is empty. Run per-sample analysis first.")

consistent_pairs = get_consistent_pairs(df_pairs_all, focus_labels=["T", "B"])
display(consistent_pairs.head(20))

In [ ]:
# Consensus heatmaps

if df_pairs_all.empty:
    raise ValueError("df_pairs_all is empty. Run per-sample analysis first.")

min_samples_sig = 0.5  # >=50% samples significant

# Build pair set for circle overlays
pair_set = set(zip(consistent_pairs["source"], consistent_pairs["neighbor"]))

# Build consensus table and save summary CSV
consensus = build_consensus_table(
    df_pairs_all,
    min_samples_sig=min_samples_sig,
    include_dispersion=True,
    out_csv=f"{outdir}nhood_enrichment_consensus_summary.csv" if outdir is not None else None,
)

# Category order
common_categories = get_common_categories(adata.obs["celltype"])

# Plot + save figures/CSVs via helper
_ = plot_consensus_heatmaps(
    consensus=consensus,
    categories=common_categories,
    min_samples_sig=min_samples_sig,
    pair_set=pair_set,
    clip=(-2, 2),
    box_text_size=FONT_SIZE * 1.8,
    title_size=FONT_SIZE * 1.8,
    axis_label_size=FONT_SIZE * 1.8,
    tick_label_size=FONT_SIZE * 1.8,
    figsize_left=(FIGSIZE_IN[0] * 2, FIGSIZE_IN[1] * 2),
    figsize_right=(FIGSIZE_IN[0] * 2, FIGSIZE_IN[1] * 2),
    cluster=True,
    show_dendrogram=True,
    use_symmetric_for_order=True,
    cluster_method="average",
    cluster_metric="euclidean",
    left_title="",
    right_title="Mean Observed Neighbor Counts",
    figdir=figdir,
    outdir=outdir,
    filename_prefix="nhood_enrichment_",
)

### Consensus for timepoints (main celltypes)

In [ ]:
# Consensus for timepoints (main celltypes) using unified helper functions

if df_pairs_all.empty:
    raise ValueError("df_pairs_all is empty. Run per-sample analysis first.")

timepoint_key = "timepoint"
sample_key = "sample"
min_samples_sig = 0.5

# Ensure consistent_pairs exists (used for circle overlays)
consistent_pairs = get_consistent_pairs(df_pairs_all, focus_labels=["T", "B"])
pair_set = set(zip(consistent_pairs["source"], consistent_pairs["neighbor"]))

# consistent ordering
present = set(df_pairs_all["source"]).union(set(df_pairs_all["neighbor"]))
common_categories = [c for c in celltype_order if c in present] + [c for c in sorted(present) if c not in celltype_order]

# attach timepoint to df_pairs_all
sample_to_timepoint = adata.obs[[sample_key, timepoint_key]].drop_duplicates()
df_pairs_tp = df_pairs_all.merge(sample_to_timepoint, on=sample_key, how="left")

for tp in sorted(df_pairs_tp[timepoint_key].dropna().unique()):
    df_tp = df_pairs_tp[df_pairs_tp[timepoint_key] == tp].copy()

    # consensus summary per timepoint
    consensus_tp = build_consensus_table(
        df_pairs=df_tp,
        min_samples_sig=min_samples_sig,
        include_dispersion=True,
        out_csv=f"{outdir}nhood_enrichment_consensus_{tp}.csv" if outdir is not None else None,
    )

    # plot/save via helper
    _ = plot_consensus_heatmaps(
        consensus=consensus_tp,
        categories=common_categories,
        min_samples_sig=min_samples_sig,
        pair_set=pair_set,
        clip=(-2, 2),
        box_text_size=FONT_SIZE,
        title_size=FONT_SIZE,
        axis_label_size=FONT_SIZE,
        tick_label_size=FONT_SIZE,
        figsize_left=FIGSIZE_IN,
        figsize_right=FIGSIZE_IN,
        cluster=False,            # matches old behavior (no dendrogram)
        show_dendrogram=False,
        use_symmetric_for_order=True,
        cluster_method="average",
        cluster_metric="euclidean",
        left_title=f"Consensus Enrichment ({tp})",
        right_title=f"Mean Observed Neighbor Counts ({tp})",
        figdir=figdir,
        outdir=outdir,
        filename_prefix=f"nhood_enrichment_{tp}_",
    )

## 3D heatmaps

In [ ]:
# helpers
def val_to_color(val, norm, cmap=mcm.RdBu_r):
    """Map a signed float to an rgb(...) string via RdBu_r."""
    rgba = cmap(norm(val))
    r, g, b = [int(c * 255) for c in rgba[:3]]
    return f"rgb({r},{g},{b})"

def add_full_bar(fig, xi, yi, h, color):
    """Draw a full rectangular bar at grid position (xi, yi) with height h."""
    x0, x1 = xi - 0.4, xi + 0.4
    y0, y1 = yi - 0.4, yi + 0.4
    vx = [x0, x1, x1, x0,  x0, x1, x1, x0]
    vy = [y0, y0, y1, y1,  y0, y0, y1, y1]
    vz = [0,  0,  0,  0,   h,  h,  h,  h ]
    i_idx = [0, 0,  4, 4,  0, 0,  2, 2,  0, 0,  1, 1]
    j_idx = [1, 2,  5, 6,  1, 5,  3, 7,  3, 7,  2, 6]
    k_idx = [2, 3,  6, 7,  5, 4,  7, 6,  7, 4,  6, 5]
    fig.add_trace(go.Mesh3d(
        x=vx, y=vy, z=vz,
        i=i_idx, j=j_idx, k=k_idx,
        color=color, opacity=1,
        flatshading=True,
        lighting=dict(ambient=0.9, diffuse=0.1),
        showscale=False,
    ))

def add_bar_edges(fig, xi, yi, h, color="black", width=1):
    """Draw wireframe outline of a rectangular bar."""
    x0, x1 = xi - 0.4, xi + 0.4
    y0, y1 = yi - 0.4, yi + 0.4
    corners = [(x0,y0), (x1,y0), (x1,y1), (x0,y1)]
    ex, ey, ez = [], [], []
    for z_level in [0, h]:
        for i in range(4):
            j = (i + 1) % 4
            ex += [corners[i][0], corners[j][0], None]
            ey += [corners[i][1], corners[j][1], None]
            ez += [z_level, z_level, None]
    for cx, cy in corners:
        ex += [cx, cx, None]
        ey += [cy, cy, None]
        ez += [0, h, None]
    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez, mode="lines",
        line=dict(color=color, width=width),
        showlegend=False, hoverinfo="none",
    ))

def add_highlight_ring(fig, xi, yi, h, color="black", radius=0.38, width=4, n_pts=80):
    """Draw a hollow ring lying flat on the top face of a bar."""
    theta = np.linspace(0, 2 * np.pi, n_pts)
    fig.add_trace(go.Scatter3d(
        x=(xi + radius * np.cos(theta)).tolist(),
        y=(yi + radius * np.sin(theta)).tolist(),
        z=[h + 0.01] * n_pts,
        mode="lines",
        line=dict(color=color, width=width),
        showlegend=False, hoverinfo="none",
    ))

In [ ]:
# load data again (if needed) and make heatmap dataframe
df = pd.read_csv(f"{outdir}nhood_enrichment_consensus_log2oe_stats.csv")

cats = ["Neuroblast", "B", "T", "Macrophage", "Fibroblast", "Endothelial", "Schwann"]
cats = [c for c in cats if c in df["source"].unique()]
n = len(cats)

mat_log2 = (
    df.pivot(index="source", columns="neighbor", values="mean_log2_oe")
    .reindex(index=cats, columns=cats).fillna(0).values
)
mat_pct = (
    df.pivot(index="source", columns="neighbor", values="pct_sig")
    .reindex(index=cats, columns=cats).fillna(0).values
)

xpos, ypos = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
xpos = xpos.flatten()
ypos = ypos.flatten()
dz     = mat_log2.flatten()
dz_pct = mat_pct.flatten()

src_labels = [cats[xi] for xi in xpos]
nbr_labels = [cats[yi] for yi in ypos]

_pos = dz[dz > 0]
_neg = dz[dz < 0]
vmax_data = float(_pos.max()) if len(_pos) else 1.0
vmin_data = float(_neg.min()) if len(_neg) else -1.0
norm = mcolors.TwoSlopeNorm(vmin=vmin_data, vcenter=0, vmax=vmax_data)

In [ ]:
# shared layout
_elev, _azim = 40, -215
_r = 4
_eye = dict(
    x = _r * math.cos(math.radians(_elev)) * math.cos(math.radians(_azim)),
    y = _r * math.cos(math.radians(_elev)) * math.sin(math.radians(_azim)),
    z = _r * math.sin(math.radians(_elev)),
)

_font = dict(family="Helvetica", size=5, color="black")
axis_style = dict(
    showbackground=False, showgrid=False, zeroline=False,
    showline=True, linecolor="black", tickcolor="black", tickfont=_font,
)

tick_vals = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

In [ ]:
# two panel plots
panels = [
    ("Enriched (positive)", dz > 0,  [("B", "T"), ("B", "B"), ("T", "T"), ("T", "B")]),
    ("Depleted (negative)", dz < 0,  [("B", "Neuroblast"), ("Neuroblast", "B"), ("T", "Neuroblast"), ("Neuroblast", "T")]),
]

for title_suffix, mask, highlight_pairs in panels:
    fig = go.Figure()

    for i in np.where(mask)[0]:
        xi, yi = int(xpos[i]), int(ypos[i])
        val = float(dz[i])
        h   = float(np.abs(val))
        add_full_bar(fig, xi, yi, h, val_to_color(val, norm))
        add_bar_edges(fig, xi, yi, h)

    for src, nbr in highlight_pairs:
        if src not in cats or nbr not in cats:
            continue
        xi = cats.index(src)
        yi = cats.index(nbr)
        h  = float(np.abs(dz[xi * n + yi]))
        add_highlight_ring(fig, xi, yi, h, color="white", radius=0.2, width=6)

    tick_texts = (
        ["0", "-0.5", "-1.0", "-1.5", "-2.0", "-2.5", "-3.0", "-3.5"]
        if "negative" in title_suffix.lower()
        else [str(v) for v in tick_vals]
    )

    fig.update_layout(
        title=f"Consensus Neighborhood Enrichment — {title_suffix}",
        scene=dict(
            aspectmode="manual",
            aspectratio=dict(x=1.2, y=1.2, z=1.0),
            xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Source", font=_font)),
            yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Neighbor", font=_font)),
            zaxis=dict(**axis_style, title=dict(text="Mean log2 O/E", font=_font),
                       range=[0, 3.5], tickvals=tick_vals, ticktext=tick_texts),
            camera=dict(eye=_eye),
            bgcolor="white",
        ),
        paper_bgcolor="white", plot_bgcolor="white",
        width=800, height=950,
    )
    fig.write_image(f"{figdir}nhood_enrichment_{title_suffix}.png", width=2000, height=2000, scale=1)
    fig.show()

In [ ]:
# combined plot

highlight_pairs_combined = [
    ("B", "T"), ("B", "B"), ("T", "T"), ("T", "B"),
    ("B", "Neuroblast"), ("Neuroblast", "B"), ("T", "Neuroblast"), ("Neuroblast", "T"),
]

fig = go.Figure()

for i in np.where(dz != 0)[0]:
    xi, yi = int(xpos[i]), int(ypos[i])
    val = float(dz[i])
    h   = float(np.abs(val))
    add_full_bar(fig, xi, yi, h, val_to_color(val, norm))
    add_bar_edges(fig, xi, yi, h)

for src, nbr in highlight_pairs_combined:
    if src not in cats or nbr not in cats:
        continue
    xi = cats.index(src)
    yi = cats.index(nbr)
    h  = float(np.abs(dz[xi * n + yi]))
    if h > 0:
        add_highlight_ring(fig, xi, yi, h, color="white", radius=0.2, width=6)

fig.update_layout(
    title="Consensus Neighborhood Enrichment — Combined",
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=1.2, y=1.2, z=1.0),
        xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                   title=dict(text="Source", font=_font)),
        yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                   title=dict(text="Neighbor", font=_font)),
        zaxis=dict(**axis_style, title=dict(text="|Mean log2 O/E|", font=_font),
                   range=[0, 3.5], tickvals=tick_vals, ticktext=[str(v) for v in tick_vals]),
        camera=dict(eye=_eye),
        bgcolor="white",
    ),
    paper_bgcolor="white", plot_bgcolor="white",
    width=800, height=950,
)
fig.write_image(f"{figdir}nhood_enrichment_combined.png", width=2000, height=2000, scale=1)
fig.show()

In [ ]:
# per-sample (new)
_font = dict(family="Helvetica", size=1, color="white")
axis_style = dict(
    showbackground=False, showgrid=False, zeroline=False,
    showline=True, linecolor="black", tickcolor="black", tickfont=_font,
)

df_all = pd.read_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv")

cats = ["Neuroblast", "B", "T", "Macrophage", "Fibroblast", "Endothelial", "Schwann"]
cats = [c for c in cats if c in df_all["source"].unique()]
n = len(cats)

highlight_pairs_combined = [
    ("B", "T"), ("B", "B"), ("T", "T"), ("T", "B"),
    ("B", "Neuroblast"), ("Neuroblast", "B"), ("T", "Neuroblast"), ("Neuroblast", "T"),
]

grey_overrides = {
    ("Patient3_PT", "B", "B"),
    ("Patient4_PT", "B", "B"),
    ("Patient5_PT", "B", "B"),
}

# global norm across all samples for a consistent color scale
_all_vals = df_all["log2_oe"].values
_pos_all  = _all_vals[_all_vals > 0]
_neg_all  = _all_vals[_all_vals < 0]
vmax_data = float(_pos_all.max()) if len(_pos_all) else 1.0
vmin_data = float(_neg_all.min()) if len(_neg_all) else -1.0
norm = mcolors.TwoSlopeNorm(vmin=vmin_data, vcenter=0, vmax=vmax_data)

z_max = 3.5
tick_vals = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

for sample in sorted(df_all["sample"].unique()):
    df_s = df_all[df_all["sample"] == sample]

    mat_log2 = (
        df_s.pivot(index="source", columns="neighbor", values="log2_oe")
        .reindex(index=cats, columns=cats).fillna(0).values
    )

    xpos, ypos = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    xpos = xpos.flatten()
    ypos = ypos.flatten()
    dz   = mat_log2.flatten()

    fig = go.Figure()

    for i in np.where(dz != 0)[0]:
        xi, yi = int(xpos[i]), int(ypos[i])
        val = float(dz[i])
        h   = float(np.abs(val))
        if (sample, cats[xi], cats[yi]) in grey_overrides:
            bar_color  = "white"
            edge_color = "white"
        else:
            bar_color  = val_to_color(val, norm)
            edge_color = "black"
        add_full_bar(fig, xi, yi, h, bar_color)
        add_bar_edges(fig, xi, yi, h, color=edge_color)

    for src, nbr in highlight_pairs_combined:
        if src not in cats or nbr not in cats:
            continue
        xi = cats.index(src)
        yi = cats.index(nbr)
        h  = float(np.abs(dz[xi * n + yi]))
        if h > 0:
            h_ring = min(h + 0.01, z_max - 0.05)
            add_highlight_ring(fig, xi, yi, h_ring, color="white", radius=0.2, width=6)

    fig.update_layout(
        title=dict(text=f"Neighborhood Enrichment — {sample}", font=_font),
        scene=dict(
            aspectmode="manual",
            aspectratio=dict(x=1.2, y=1.2, z=1.0),
            xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Source", font=_font)),
            yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Neighbor", font=_font)),
            zaxis=dict(**axis_style, title=dict(text="|log2 O/E|", font=_font),
                       range=[0, z_max], tickvals=tick_vals,
                       ticktext=[str(v) for v in tick_vals]),
            camera=dict(eye=_eye),
            bgcolor="white",
        ),
        paper_bgcolor="white", plot_bgcolor="white",
        width=800, height=950,
    )
    fig.write_image(f"{figdir}nhood_enrichment_{sample}_grey.png", width=2000, height=2000, scale=1)
    fig.show()

In [ ]:
# per-sample
_font = dict(family="Helvetica", size=1, color="white")
axis_style = dict(
    showbackground=False, showgrid=False, zeroline=False,
    showline=True, linecolor="black", tickcolor="black", tickfont=_font,
)

df_all = pd.read_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv")

cats = ["Neuroblast", "B", "T", "Macrophage", "Fibroblast", "Endothelial", "Schwann"]
cats = [c for c in cats if c in df_all["source"].unique()]
n = len(cats)

highlight_pairs_combined = [
    ("B", "T"), ("B", "B"), ("T", "T"), ("T", "B"),
    ("B", "Neuroblast"), ("Neuroblast", "B"), ("T", "Neuroblast"), ("Neuroblast", "T"),
]

# global norm across all samples for a consistent color scale
_all_vals = df_all["log2_oe"].values
_pos_all  = _all_vals[_all_vals > 0]
_neg_all  = _all_vals[_all_vals < 0]
vmax_data = float(_pos_all.max()) if len(_pos_all) else 1.0
vmin_data = float(_neg_all.min()) if len(_neg_all) else -1.0
norm = mcolors.TwoSlopeNorm(vmin=vmin_data, vcenter=0, vmax=vmax_data)

z_max = 3.5
tick_vals = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

for sample in sorted(df_all["sample"].unique()):
    df_s = df_all[df_all["sample"] == sample]

    mat_log2 = (
        df_s.pivot(index="source", columns="neighbor", values="log2_oe")
        .reindex(index=cats, columns=cats).fillna(0).values
    )

    xpos, ypos = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    xpos = xpos.flatten()
    ypos = ypos.flatten()
    dz   = mat_log2.flatten()

    fig = go.Figure()

    for i in np.where(dz != 0)[0]:
        xi, yi = int(xpos[i]), int(ypos[i])
        val = float(dz[i])
        h   = float(np.abs(val))
        add_full_bar(fig, xi, yi, h, val_to_color(val, norm))
        add_bar_edges(fig, xi, yi, h)

    for src, nbr in highlight_pairs_combined:
        if src not in cats or nbr not in cats:
            continue
        xi = cats.index(src)
        yi = cats.index(nbr)
        h  = float(np.abs(dz[xi * n + yi]))
        if h > 0:
            h_ring = min(h + 0.01, z_max - 0.05)
            add_highlight_ring(fig, xi, yi, h_ring, color="white", radius=0.2, width=6)

    fig.update_layout(
        title=dict(text=f"Neighborhood Enrichment — {sample}", font=_font),
        scene=dict(
            aspectmode="manual",
            aspectratio=dict(x=1.2, y=1.2, z=1.0),
            xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Source", font=_font)),
            yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats,
                       title=dict(text="Neighbor", font=_font)),
            zaxis=dict(**axis_style, title=dict(text="|log2 O/E|", font=_font),
                       range=[0, z_max], tickvals=tick_vals,
                       ticktext=[str(v) for v in tick_vals]),
            camera=dict(eye=_eye),
            bgcolor="white",
        ),
        paper_bgcolor="white", plot_bgcolor="white",
        width=800, height=950,
    )
    fig.write_image(f"{figdir}nhood_enrichment_{sample}.png", width=2000, height=2000, scale=1)
    fig.show()

## Extra distance analysis
Not included in the manuscript

### Per-sample (all celltypes)

In [ ]:
# adjust font size
FONT_SIZE = 1.9

In [ ]:
# Run for all samples (celltypes_all) using unified helper functions

# Build subtype order from parent celltype order
parent_order = ["Endothelial", "Fibroblast", "Schwann", "Neuroblast", "Macrophage", "B", "T"]
subtype_order = build_subtype_order_from_parent(
    adata=adata,
    parent_order=parent_order,
    subtype_key="celltypes_all",
    parent_key="celltype",
)

prefix = "celltypes_all_"

all_df = []
samples = adata.obs["sample"].unique().tolist()

for sample in samples:
    print(f"Processing {sample}...")
    adata_sub = adata[adata.obs["sample"] == sample].copy()
    if adata_sub.n_obs < 100:
        continue

    df_pairs = plot_nhood(
        adata_sub,
        group_key="celltypes_all",
        sample_label=f"{sample} (n={adata_sub.n_obs})",
        n_perms=5000,
        seed=0,
        alpha=0.05,
        pseudocount=1.0,
        clip=(-2, 2),
        mask_upper_triangle=False,
        coord_type="generic",
        spatial_key="spatial",
        value_fmt="{:.2f}",
        n_jobs=8,
        figdir=outdir,               # save PDFs in outdir
        outdir=outdir,               # save CSVs in outdir
        label_order=subtype_order,
        filename_prefix=prefix,
    )

    df_pairs["sample"] = sample
    all_df.append(df_pairs)

    display(
        df_pairs.query("significant and not low_expected")
                .sort_values("log2_oe", ascending=False)
                .head(20)
    )

# Save combined CSV
df_pairs_all = pd.concat(all_df, ignore_index=True) if len(all_df) else pd.DataFrame()
if not df_pairs_all.empty and outdir is not None:
    df_pairs_all.to_csv(f"{outdir}{prefix}nhood_enrichment_all_samples.csv", index=False)

### Delta (celltypes_all)

In [ ]:
timepoint_key = "timepoint"
sample_key = "sample"
patient_key = "patient"

# load celltypes_all per-sample results
df_pairs_all_cta = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")

if df_pairs_all_cta.empty:
    raise ValueError("celltypes_all results are empty. Run the celltypes_all per-sample analysis first.")

# Build subtype order if not already present
if "subtype_order" not in globals():
    parent_order = ["Endothelial", "Fibroblast", "Schwann", "Neuroblast", "Macrophage", "B", "T"]
    subtype_order = build_subtype_order_from_parent(
        adata=adata,
        parent_order=parent_order,
        subtype_key="celltypes_all",
        parent_key="celltype",
    )

# style aligned with helper cell
value_fmt = "{:.2f}"
vmin, vmax = -2, 2
figsize_delta = FIGSIZE_IN
title_size = FONT_SIZE
axis_label_size = FONT_SIZE
annot_fontsize = FONT_SIZE
tick_label_size = FONT_SIZE

# attach patient + timepoint to each sample
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()
df_pairs_meta = df_pairs_all_cta.merge(meta, on=sample_key, how="left")

# keep only DX/PT
df_pairs_meta = df_pairs_meta[df_pairs_meta[timepoint_key].isin(["DX", "PT"])].copy()

# per-patient heatmaps
for patient in sorted(df_pairs_meta[patient_key].dropna().unique()):
    df_p = df_pairs_meta[df_pairs_meta[patient_key] == patient].copy()

    dx = df_p[df_p[timepoint_key] == "DX"].set_index(["source", "neighbor"])
    pt = df_p[df_p[timepoint_key] == "PT"].set_index(["source", "neighbor"])

    # skip if either side missing
    if dx.empty or pt.empty:
        continue

    # only keep pairs present in both
    paired = pt[["log2_oe"]].rename(columns={"log2_oe": "pt"}).join(
        dx[["log2_oe"]].rename(columns={"log2_oe": "dx"}),
        how="inner"
    )
    paired["delta"] = paired["pt"] - paired["dx"]
    delta = paired["delta"].reset_index()

    # matrix in subtype order
    df_delta = delta.pivot(index="source", columns="neighbor", values="delta").reindex(
        index=subtype_order, columns=subtype_order
    )
    df_delta_clipped = df_delta.clip(lower=vmin, upper=vmax)

    # optional outputs: values + stats CSV per heatmap
    if outdir is not None:
        df_delta.to_csv(f"{outdir}celltypes_all_delta_PT_minus_DX_{patient}_values.csv")
        delta.to_csv(f"{outdir}celltypes_all_delta_PT_minus_DX_{patient}_stats.csv", index=False)

    # plot
    fig, ax = plt.subplots(figsize=figsize_delta)
    sns.heatmap(
        df_delta_clipped,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        square=True,
        xticklabels=subtype_order,
        yticklabels=subtype_order,
        annot=False,
        linewidths=0.3,
        linecolor="lightgray",
        cbar_kws={"label": "Δ log2(Obs/Exp_perm_mean) (PT - DX)"},
        ax=ax
    )

    # manual text coloring
    for i, s in enumerate(df_delta.index):
        for j, n in enumerate(df_delta.columns):
            val = df_delta.iloc[i, j]
            if pd.isna(val):
                continue
            text_color = "white" if abs(val) > 1.5 else "black"
            ax.text(
                j + 0.5, i + 0.5, value_fmt.format(val),
                ha="center", va="center",
                fontsize=annot_fontsize, color=text_color
            )

    ax.set_title(f"PT - DX Delta\n(Patient {patient}) - celltypes_all", fontsize=title_size)
    ax.set_xlabel("Neighbor cell type", fontsize=axis_label_size)
    ax.set_ylabel("Source cell type", fontsize=axis_label_size)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center", fontsize=tick_label_size)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, va="center", fontsize=tick_label_size)
    _apply_heatmap_tick_style(ax)

    plt.tight_layout()
    save_dir = figdir if figdir is not None else outdir
    if save_dir is not None:
        plt.savefig(
            f"{save_dir}celltypes_all_delta_PT_minus_DX_{patient}.pdf",
            format="pdf",
            bbox_inches="tight",
            transparent=True
        )
    plt.show()
    plt.close(fig)

### Dendogram consensus (celltypes_all)

In [ ]:
# Consistency of sign among significant samples (celltypes_all, T/B by parent type)

df_pairs_all_cta = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")
if df_pairs_all_cta.empty:
    raise ValueError("celltypes_all_nhood_enrichment_all_samples.csv is empty.")

sig = df_pairs_all_cta.query("significant and not low_expected").copy()

# Build subtype -> parent mapping (also include parent->parent identity)
df_map = adata.obs[["celltypes_all", "celltype"]].dropna().drop_duplicates()
subtype_to_parent = dict(zip(df_map["celltypes_all"], df_map["celltype"]))
for p in df_map["celltype"].unique():
    subtype_to_parent[p] = p

sig["parent_source"] = sig["source"].map(lambda x: subtype_to_parent.get(x, x))
sig["parent_neighbor"] = sig["neighbor"].map(lambda x: subtype_to_parent.get(x, x))

# Keep only pairs involving T or B at parent level
sig_tb = sig[
    sig["parent_source"].isin(["T", "B"]) | sig["parent_neighbor"].isin(["T", "B"])
].copy()

consistency = sig_tb.groupby(["source", "neighbor"])["log2_oe"].agg(
    n_sig="size",
    all_same_sign=sign_consistency,
    direction=lambda x: "enriched" if (x > 0).all() else ("depleted" if (x < 0).all() else "mixed")
).reset_index()

consistent_pairs_all = consistency[consistency["all_same_sign"]].copy()
display(consistent_pairs_all)

In [ ]:
# Consensus heatmaps (celltypes_all)

df_pairs_all_cta = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")
if df_pairs_all_cta.empty:
    raise ValueError("celltypes_all results are empty. Run the celltypes_all per-sample analysis first.")

# Ensure consistent_pairs_all exists (from your celltypes_all consistency cell)
if "consistent_pairs_all" not in globals():
    raise ValueError("consistent_pairs_all is not defined. Run the celltypes_all consistency cell first.")

min_samples_sig = 0.5

# Build pair set for circle overlays
pair_set = set(zip(consistent_pairs_all["source"], consistent_pairs_all["neighbor"]))

# Build consensus table and save summary CSV
consensus_cta = build_consensus_table(
    df_pairs=df_pairs_all_cta,
    min_samples_sig=min_samples_sig,
    include_dispersion=True,
    out_csv=f"{outdir}celltypes_all_nhood_enrichment_consensus_summary.csv" if outdir is not None else None,
)

# Category order: subtype_order first if available, then leftovers
if "subtype_order" in globals():
    cta_categories = [x for x in subtype_order if x in pd.unique(df_pairs_all_cta["source"])]
    leftovers = [x for x in pd.unique(df_pairs_all_cta["source"]) if x not in cta_categories]
    cta_categories = cta_categories + sorted(leftovers)
else:
    cta_categories = sorted(pd.unique(df_pairs_all_cta["source"]))

# Plot + save figures/CSVs via helper (with dendrogram)
_ = plot_consensus_heatmaps(
    consensus=consensus_cta,
    categories=cta_categories,
    min_samples_sig=min_samples_sig,
    pair_set=pair_set,
    clip=(-2, 2),
    box_text_size=FONT_SIZE,
    title_size=FONT_SIZE,
    axis_label_size=FONT_SIZE,
    tick_label_size=FONT_SIZE,
    figsize_left=FIGSIZE_IN,
    figsize_right=FIGSIZE_IN,
    cluster=True,
    show_dendrogram=True,
    use_symmetric_for_order=True,
    cluster_method="average",
    cluster_metric="euclidean",
    left_title="Consensus Enrichment (celltypes_all)",
    right_title="Mean Observed Neighbor Counts (celltypes_all)",
    figdir=figdir,
    outdir=outdir,
    filename_prefix="celltypes_all_",
)

### Consensus for timepoints (celltypes_all)

In [ ]:
timepoint_key = "timepoint"
sample_key = "sample"
min_samples_sig = 0.5
prefix = "celltypes_all_"

# Load celltypes_all per-sample results
df_pairs_all_cta = pd.read_csv(f"{outdir}{prefix}nhood_enrichment_all_samples.csv")
if df_pairs_all_cta.empty:
    raise ValueError("celltypes_all results are empty. Run the celltypes_all per-sample analysis first.")

# subtype order needed for consistent ordering
if "subtype_order" not in globals():
    raise ValueError("subtype_order is not defined. Run the subtype_order cell first.")

# attach timepoint metadata
sample_to_timepoint = adata.obs[[sample_key, timepoint_key]].drop_duplicates()
df_pairs_tp = df_pairs_all_cta.merge(sample_to_timepoint, on=sample_key, how="left")

if df_pairs_tp[timepoint_key].isna().any():
    raise ValueError("Some samples are missing timepoint annotations.")

# optional circle overlays
pair_set = (
    set(zip(consistent_pairs_all["source"], consistent_pairs_all["neighbor"]))
    if "consistent_pairs_all" in globals() else set()
)

for tp in sorted(df_pairs_tp[timepoint_key].dropna().unique()):
    df_tp = df_pairs_tp[df_pairs_tp[timepoint_key] == tp].copy()

    # consensus table for this timepoint
    consensus_tp = build_consensus_table(
        df_pairs=df_tp,
        min_samples_sig=min_samples_sig,
        include_dispersion=True,
        out_csv=f"{outdir}{prefix}nhood_enrichment_consensus_{tp}.csv" if outdir is not None else None,
    )

    # keep subtype_order first, then any leftover labels
    present = pd.unique(pd.concat([df_tp["source"], df_tp["neighbor"]], axis=0))
    ordered = [x for x in subtype_order if x in present]
    leftovers = [x for x in present if x not in ordered]
    cta_categories = ordered + sorted(leftovers)

    _ = plot_consensus_heatmaps(
        consensus=consensus_tp,
        categories=cta_categories,
        min_samples_sig=min_samples_sig,
        pair_set=pair_set,
        clip=(-2, 2),
        box_text_size=FONT_SIZE,
        title_size=FONT_SIZE,
        axis_label_size=FONT_SIZE,
        tick_label_size=FONT_SIZE,
        figsize_left=FIGSIZE_IN,
        figsize_right=FIGSIZE_IN,
        cluster=False,            # matches prior non-dendrogram timepoint behavior
        show_dendrogram=False,
        use_symmetric_for_order=True,
        cluster_method="average",
        cluster_metric="euclidean",
        left_title=f"Consensus Enrichment ({tp})",
        right_title=f"Consensus Observed Neighbor Counts ({tp})",
        figdir=figdir,
        outdir=outdir,
        filename_prefix=f"{prefix}nhood_enrichment_consensus_{tp}_",
    )

## Extra code chunks for additional visualization
Not included in the manuscript

In [ ]:
# Compute cell type proportions across timepoints

def compute_proportions(df, group_col):
    """
    Compute proportions of group_col for each patient-timepoint combination.
    Returns: DataFrame with columns [patient, timepoint, group_col, proportion]
    """
    # Count cells per group within each patient-timepoint
    counts = df.groupby(['patient', 'timepoint', group_col]).size().reset_index(name='count')
    
    # Get total cells per patient-timepoint
    totals = df.groupby(['patient', 'timepoint']).size().reset_index(name='total')
    
    # Merge and compute proportions
    props = counts.merge(totals, on=['patient', 'timepoint'])
    props['proportion'] = (props['count'] / props['total']) * 100
    
    return props[['patient', 'timepoint', group_col, 'proportion']]

def test_dx_vs_pt(props_df, group_col):
    """
    Paired Wilcoxon test comparing DX vs PT for each level of group_col.
    
    Parameters:
    -----------
    props_df : DataFrame with columns [patient, timepoint, group_col, proportion]
    group_col : str, name of the grouping column
    
    Returns:
    --------
    DataFrame with test results
    """
    results = []
    
    for group_val in props_df[group_col].unique():
        # Get data for this group
        group_data = props_df[props_df[group_col] == group_val]
        
        # Pivot to get DX and PT columns
        pivot = group_data.pivot_table(
            index='patient',
            columns='timepoint',
            values='proportion',
            aggfunc='first'
        )
        
        # Check if we have both DX and PT
        if 'DX' in pivot.columns and 'PT' in pivot.columns:
            # Remove patients with missing data
            pivot = pivot.dropna()
            
            if len(pivot) >= 2:  # Need at least 2 pairs
                try:
                    stat, p_val = wilcoxon(pivot['DX'], pivot['PT'], alternative='two-sided')
                    median_dx = pivot['DX'].median()
                    median_pt = pivot['PT'].median()
                    delta = median_pt - median_dx
                    
                    results.append({
                        group_col: group_val,
                        'n_pairs': len(pivot),
                        'median_DX': median_dx,
                        'median_PT': median_pt,
                        'delta_PT_minus_DX': delta,
                        'p_value': p_val
                    })
                except:
                    pass  # Skip if test fails (e.g., all zeros)
    
    # Convert to DataFrame and add FDR correction
    if results:
        results_df = pd.DataFrame(results)
        
        # Apply proper Benjamini-Hochberg FDR correction
        reject, pvals_corrected, alphacSidak, alphacBonf = multipletests(
            results_df['p_value'], 
            alpha=0.05, 
            method='fdr_bh'
        )
        
        results_df['FDR'] = pvals_corrected
        results_df['significant'] = reject
        
        return results_df.sort_values('p_value')
    else:
        return pd.DataFrame()

def plot_paired_proportions(props_df, group_col, group_value, stats_df=None, figsize=None):
    """
    Create paired line plot for DX -> PT changes.
    
    Parameters:
    -----------
    props_df : DataFrame from compute_proportions()
    group_col : str, column name ('celltype' or 'Multinomial_Label')
    group_value : str, specific value to plot
    stats_df : DataFrame, results from test_dx_vs_pt() with FDR correction
    figsize : tuple, figure size (default: (5.5, 4))
    """
    
    # Filter to specific group
    data = props_df[props_df[group_col] == group_value].copy()
    
    if len(data) == 0:
        print(f"No data for {group_value}")
        return None
    
    # Ensure timepoint order
    data['timepoint'] = pd.Categorical(data['timepoint'], categories=['DX', 'PT'], ordered=True)
    data = data.sort_values('timepoint')
    
    # Single plot
    if figsize is None:
        figsize = (5.5, 4)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot individual patient trajectories, colored by patient
    for patient in data['patient'].unique():
        patient_data = data[data['patient'] == patient].sort_values('timepoint')
        if len(patient_data) == 2:
            ax.plot(patient_data['timepoint'], patient_data['proportion'],
                   'o-', alpha=1, linewidth=2.5, markersize=8,
                   color=patient_colors.get(patient, 'gray'),
                   label=patient)
    
    ax.set_xlabel('Timepoint', fontsize=14, fontweight='bold')
    ax.set_ylabel('Proportion (%)', fontsize=14, fontweight='bold')
    ax.set_title(group_value, fontsize=15, fontweight='bold')
    ax.tick_params(labelsize=12, left=False, bottom=False, labelleft=True, labelbottom=True)
    ax.grid(False)
    ax.set_frame_on(False)
    
    # Legend on the right
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), 
             fontsize=10, frameon=False)
    
    # Add statistics including FDR
    if stats_df is not None and not stats_df.empty:
        stat_row = stats_df[stats_df[group_col] == group_value]
        
        if not stat_row.empty:
            s = stat_row.iloc[0]
            
            # Build text with p-value and FDR
            text = f"p = {s['p_value']:.4f}\n"
            
            if 'FDR' in s:
                text += f"FDR = {s['FDR']:.4f}"
                
                # Add significance marker if FDR < 0.05
                if s['FDR'] < 0.05:
                    text += " *"
            
            # Display delta

            if 'delta_PT_minus_DX' in s:
                text += f"\nΔ: {s['delta_PT_minus_DX']:.2f}%"
            
            ax.text(0.5, 0.95, text, transform=ax.transAxes,
                   fontsize=10, verticalalignment='top',
                   horizontalalignment='center',
                            )
    
    plt.tight_layout()
    plt.show()
    
    return fig

In [ ]:
print("\n" + "="*80)
print("Main celltypes proportions DX vs PT")
print("="*80)
celltype_main_props = compute_proportions(adata.obs, 'celltype')
celltype_main = test_dx_vs_pt(celltype_main_props, 'celltype')
print(celltype_main.to_string())

print("\n" + "="*80)
print("All celltypes proportions DX vs PT")
print("="*80)
celltype_all_props = compute_proportions(adata.obs, 'celltypes_all')
celltype_all = test_dx_vs_pt(celltype_all_props, 'celltypes_all')
print(celltype_all.to_string())

# Plot cell types of interest
# c1qc_macro = plot_paired_proportions(celltype_all_props, 'celltypes_all', 'C1QC Mφ', stats_df=celltype_all)

In [ ]:
# Top20 robust heterotypic pairs involving T or B
# for celltype + celltypes_all

def compute_topN_robust_TB(df_pairs_all, label_type="celltypes_all",
                           patient_key="patient", sample_key="sample",
                           min_samples_sig=0.5, min_patients=2, top_n=20):
    if df_pairs_all.empty:
        raise ValueError("df_pairs_all is empty.")

    # reproducibility summary
    df_sig = df_pairs_all.query("significant and not low_expected").copy()
    total_samples = df_pairs_all.groupby(["source","neighbor"])[sample_key].nunique().reset_index(name="n_samples_total")
    sig_samples = df_sig.groupby(["source","neighbor"])[sample_key].nunique().reset_index(name="n_samples_sig")
    effect_stats = df_pairs_all.groupby(["source","neighbor"])["log2_oe"].agg(
        mean_log2_oe="mean",
        sd_log2_oe="std",
        iqr_log2_oe=lambda x: np.nanpercentile(x, 75) - np.nanpercentile(x, 25)
    ).reset_index()

    summary = total_samples.merge(sig_samples, on=["source","neighbor"], how="left")
    summary["n_samples_sig"] = summary["n_samples_sig"].fillna(0).astype(int)
    summary["pct_samples_sig"] = 100 * summary["n_samples_sig"] / summary["n_samples_total"]
    summary = summary.merge(effect_stats, on=["source","neighbor"], how="left")

    # patient reproducibility
    sample_to_patient = adata.obs[[sample_key, patient_key]].drop_duplicates()
    df_pairs_all = df_pairs_all.merge(sample_to_patient, on=sample_key, how="left")

    if patient_key in df_pairs_all.columns:
        df_sig = df_pairs_all.query("significant and not low_expected").copy()
        n_patients = df_sig.groupby(["source","neighbor"])[patient_key].nunique().reset_index(name="n_patients_sig")
        summary = summary.merge(n_patients, on=["source","neighbor"], how="left")
        summary["n_patients_sig"] = summary["n_patients_sig"].fillna(0).astype(int)
    else:
        summary["n_patients_sig"] = np.nan

    # robust filter
    robust = summary[
        (summary["pct_samples_sig"] >= (min_samples_sig * 100)) &
        (summary["n_patients_sig"] >= min_patients)
    ].copy()

    # parent mapping for category labelser
    df_map = adata.obs[["celltypes_all","celltype"]].dropna().drop_duplicates()
    def _parent_type(x):
        if x in df_map["celltype"].unique():
            return x
        hit = df_map.loc[df_map["celltypes_all"] == x, "celltype"]
        return hit.iloc[0] if len(hit) else x

    robust["parent_source"] = robust["source"].map(_parent_type)
    robust["parent_neighbor"] = robust["neighbor"].map(_parent_type)

    # heterotypic only
    robust = robust[robust["source"] != robust["neighbor"]].copy()

    # keep only pairs involving T or B (parent types)
    robust = robust[
        (robust["parent_source"].isin(["T", "B"])) |
        (robust["parent_neighbor"].isin(["T", "B"]))
    ].copy()

    # Top N by |mean_log2_oe|
    robust["abs_log2_oe"] = robust["mean_log2_oe"].abs()
    topN = robust.sort_values("abs_log2_oe", ascending=False).head(top_n).copy()

    topN["direction"] = np.where(topN["mean_log2_oe"] >= 0, "enriched", "depleted")
    topN["label_type"] = label_type

    return robust, topN


# ---- Load data
df_celltype = pd.read_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv")
df_celltypes_all = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")

robust_celltype_TB, top20_celltype_TB = compute_topN_robust_TB(df_celltype, label_type="celltype", top_n=20)
robust_all_TB, top20_all_TB = compute_topN_robust_TB(df_celltypes_all, label_type="celltypes_all", top_n=20)

# Save both
top20_celltype_TB.to_csv(f"{outdir}top20_robust_celltype_TB.csv", index=False)
top20_all_TB.to_csv(f"{outdir}top20_robust_celltypes_all_TB.csv", index=False)

display(top20_celltype_TB[[
    "source","neighbor","parent_source","parent_neighbor",
    "direction","mean_log2_oe","n_samples_sig","n_patients_sig"
]])
display(top20_all_TB[[
    "source","neighbor","parent_source","parent_neighbor",
    "direction","mean_log2_oe","n_samples_sig","n_patients_sig"
]])

In [ ]:
# Top20 robust HOMOTYPIC pairs
# for celltype + celltypes_all

def compute_topN_robust_homotypic(df_pairs_all, label_type="celltypes_all",
                                  patient_key="patient", sample_key="sample",
                                  min_samples_sig=0.5, min_patients=2, top_n=20):
    if df_pairs_all.empty:
        raise ValueError("df_pairs_all is empty.")

    # reproducibility summary
    df_sig = df_pairs_all.query("significant and not low_expected").copy()
    total_samples = df_pairs_all.groupby(["source","neighbor"])[sample_key].nunique().reset_index(name="n_samples_total")
    sig_samples = df_sig.groupby(["source","neighbor"])[sample_key].nunique().reset_index(name="n_samples_sig")
    effect_stats = df_pairs_all.groupby(["source","neighbor"])["log2_oe"].agg(
        mean_log2_oe="mean",
        sd_log2_oe="std",
        iqr_log2_oe=lambda x: np.nanpercentile(x, 75) - np.nanpercentile(x, 25)
    ).reset_index()

    summary = total_samples.merge(sig_samples, on=["source","neighbor"], how="left")
    summary["n_samples_sig"] = summary["n_samples_sig"].fillna(0).astype(int)
    summary["pct_samples_sig"] = 100 * summary["n_samples_sig"] / summary["n_samples_total"]
    summary = summary.merge(effect_stats, on=["source","neighbor"], how="left")

    # patient reproducibility
    sample_to_patient = adata.obs[[sample_key, patient_key]].drop_duplicates()
    df_pairs_all = df_pairs_all.merge(sample_to_patient, on=sample_key, how="left")

    if patient_key in df_pairs_all.columns:
        df_sig = df_pairs_all.query("significant and not low_expected").copy()
        n_patients = df_sig.groupby(["source","neighbor"])[patient_key].nunique().reset_index(name="n_patients_sig")
        summary = summary.merge(n_patients, on=["source","neighbor"], how="left")
        summary["n_patients_sig"] = summary["n_patients_sig"].fillna(0).astype(int)
    else:
        summary["n_patients_sig"] = np.nan

    # robust filter
    robust = summary[
        (summary["pct_samples_sig"] >= (min_samples_sig * 100)) &
        (summary["n_patients_sig"] >= min_patients)
    ].copy()

    # keep homotypic only
    robust = robust[robust["source"] == robust["neighbor"]].copy()

    # Top N by |mean_log2_oe|
    robust["abs_log2_oe"] = robust["mean_log2_oe"].abs()
    topN = robust.sort_values("abs_log2_oe", ascending=False).head(top_n).copy()

    topN["direction"] = np.where(topN["mean_log2_oe"] >= 0, "enriched", "depleted")
    topN["label_type"] = label_type

    return robust, topN


# ---- Load data
df_celltype = pd.read_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv")
df_celltypes_all = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")

robust_celltype_homo, top20_celltype_homo = compute_topN_robust_homotypic(df_celltype, label_type="celltype", top_n=20)
robust_all_homo, top20_all_homo = compute_topN_robust_homotypic(df_celltypes_all, label_type="celltypes_all", top_n=20)

# Save both
top20_celltype_homo.to_csv(f"{outdir}top20_robust_celltype_homotypic.csv", index=False)
top20_all_homo.to_csv(f"{outdir}top20_robust_celltypes_all_homotypic.csv", index=False)

display(top20_celltype_homo[[
    "source","neighbor","direction","mean_log2_oe","n_samples_sig","n_patients_sig"
]])
display(top20_all_homo[[
    "source","neighbor","direction","mean_log2_oe","n_samples_sig","n_patients_sig"
]])

In [ ]:
# Consistent DX>PT trends across patients (celltype + celltypes_all)

timepoint_key = "timepoint"
patient_key = "patient"
sample_key = "sample"

def consistent_timepoint_trends(df_pairs_all, label_type="celltype", top_n=20):
    if df_pairs_all.empty:
        raise ValueError("df_pairs_all is empty.")

    # attach patient + timepoint
    sample_to_meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()
    df = df_pairs_all.merge(sample_to_meta, on=sample_key, how="left")

    # keep only DX/PT
    df = df[df[timepoint_key].isin(["DX", "PT"])].copy()

    # aggregate if multiple samples per patient/timepoint
    df_agg = df.groupby([patient_key, timepoint_key, "source", "neighbor"], as_index=False)["log2_oe"].mean()

    # pivot to DX/PT per patient
    dx = df_agg[df_agg[timepoint_key] == "DX"].rename(columns={"log2_oe": "log2_oe_DX"})
    pt = df_agg[df_agg[timepoint_key] == "PT"].rename(columns={"log2_oe": "log2_oe_PT"})

    paired = dx.merge(pt, on=[patient_key, "source", "neighbor"], how="inner")
    paired["delta"] = paired["log2_oe_PT"] - paired["log2_oe_DX"]

    # summarize consistency across patients
    summary = paired.groupby(["source", "neighbor"]).agg(
        n_patients=("delta", "size"),
        n_decrease=("delta", lambda x: (x < 0).sum()),
        n_increase=("delta", lambda x: (x > 0).sum()),
        mean_delta=("delta", "mean"),
        median_delta=("delta", "median"),
        std_delta=("delta", "std")
    ).reset_index()

    # consistent decrease or increase across ALL paired patients
    summary["consistent_decrease"] = summary["n_decrease"] == summary["n_patients"]
    summary["consistent_increase"] = summary["n_increase"] == summary["n_patients"]

    # keep only consistent trends
    consistent = summary[summary["consistent_decrease"] | summary["consistent_increase"]].copy()
    consistent["direction"] = np.where(consistent["consistent_decrease"], "decrease", "increase")
    consistent["label_type"] = label_type

    # rank by absolute mean change
    consistent["abs_mean_delta"] = consistent["mean_delta"].abs()
    consistent = consistent.sort_values("abs_mean_delta", ascending=False)

    # top N per direction (optional)
    top_decrease = consistent[consistent["direction"] == "decrease"].head(top_n)
    top_increase = consistent[consistent["direction"] == "increase"].head(top_n)

    return consistent, top_decrease, top_increase


# ---- Load data
df_celltype = pd.read_csv(f"{outdir}celltype_nhood_enrichment_all_samples.csv")
df_celltypes_all = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")

# ---- Compute for celltype
cons_celltype, top_dec_ct, top_inc_ct = consistent_timepoint_trends(df_celltype, label_type="celltype", top_n=20)

# ---- Compute for celltypes_all
cons_all, top_dec_all, top_inc_all = consistent_timepoint_trends(df_celltypes_all, label_type="celltypes_all", top_n=20)

# ---- Save
cons_celltype.to_csv(f"{outdir}consistent_trends_celltype.csv", index=False)
top_dec_ct.to_csv(f"{outdir}top20_consistent_decrease_celltype.csv", index=False)
top_inc_ct.to_csv(f"{outdir}top20_consistent_increase_celltype.csv", index=False)

cons_all.to_csv(f"{outdir}consistent_trends_celltypes_all.csv", index=False)
top_dec_all.to_csv(f"{outdir}top20_consistent_decrease_celltypes_all.csv", index=False)
top_inc_all.to_csv(f"{outdir}top20_consistent_increase_celltypes_all.csv", index=False)

# ---- Display
display(top_dec_ct[["source","neighbor","n_patients","mean_delta","median_delta","direction"]])
display(top_inc_ct[["source","neighbor","n_patients","mean_delta","median_delta","direction"]])

display(top_dec_all[["source","neighbor","n_patients","mean_delta","median_delta","direction"]])
display(top_inc_all[["source","neighbor","n_patients","mean_delta","median_delta","direction"]])

In [ ]:
# Significant-only barplot (celltypes_all, T/B pairs) with IQR (asymmetric, safe)
# Matches significance criteria from consensus heatmap

import matplotlib as mpl
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams["axes.unicode_minus"] = False

timepoint_key = "timepoint"
patient_key = "patient"
sample_key = "sample"

df_celltypes_all = pd.read_csv(f"{outdir}celltypes_all_nhood_enrichment_all_samples.csv")

# Attach patient + timepoint FIRST (before filtering)
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()
df = df_celltypes_all.merge(meta, on=sample_key, how="left")
df = df[df[timepoint_key].isin(["DX", "PT"])].copy()

# Map subtype -> parent celltype
df_map = adata.obs[["celltypes_all","celltype"]].dropna().drop_duplicates()
def _parent_type(x):
    if x in df_map["celltype"].unique():
        return x
    hit = df_map.loc[df_map["celltypes_all"] == x, "celltype"]
    return hit.iloc[0] if len(hit) else x

df["parent_source"] = df["source"].map(_parent_type)
df["parent_neighbor"] = df["neighbor"].map(_parent_type)

# Keep only pairs involving T or B
df = df[
    (df["parent_source"].isin(["T","B"])) | (df["parent_neighbor"].isin(["T","B"]))
].copy()

# Aggregate within patient/timepoint (use ALL data, not just significant)
df_agg = df.groupby([patient_key, timepoint_key, "source", "neighbor"], as_index=False)["log2_oe"].mean()

# Pivot to DX/PT per patient
dx = df_agg[df_agg[timepoint_key] == "DX"].rename(columns={"log2_oe": "log2_oe_DX"})
pt = df_agg[df_agg[timepoint_key] == "PT"].rename(columns={"log2_oe": "log2_oe_PT"})
paired = dx.merge(pt, on=[patient_key, "source", "neighbor"], how="inner")
paired["delta"] = paired["log2_oe_PT"] - paired["log2_oe_DX"]

# Check consistency FIRST
def _all_same_sign(x):
    return (x > 0).all() or (x < 0).all()

sign_consistency = paired.groupby(["source","neighbor"])["delta"].apply(_all_same_sign).reset_index(name="all_same_sign")
consistent_pairs = sign_consistency[sign_consistency["all_same_sign"]]

# NOW filter to pairs that are BOTH consistent AND significant (in at least some samples)
sig_pairs = set(zip(
    df_celltypes_all.query("significant and not low_expected")["source"],
    df_celltypes_all.query("significant and not low_expected")["neighbor"]
))

consistent_pair_set = set(zip(consistent_pairs["source"], consistent_pairs["neighbor"]))
both_sig_and_consistent = consistent_pair_set & sig_pairs

# Filter summary to only pairs that are both
summary = paired.groupby(["source","neighbor"]).agg(
    n_patients=("delta","size"),
    mean_delta=("delta","mean"),
    median_delta=("delta","median"),
    q25=("delta", lambda x: np.nanpercentile(x, 25)),
    q75=("delta", lambda x: np.nanpercentile(x, 75))
).reset_index()

summary = summary[summary.apply(lambda row: (row["source"], row["neighbor"]) in both_sig_and_consistent, axis=1)].copy()

# Build labels (ASCII arrow)
summary["pair"] = summary["source"] + " > " + summary["neighbor"]

# Sort by mean_delta
summary = summary.sort_values("mean_delta")

# Check if we have any data
if len(summary) == 0:
    print("    No pairs found that are both significant AND consistent")
    print(f"   Consistent pairs: {len(consistent_pairs)}")
    print(f"   Significant pairs: {len(sig_pairs)}")
    print(f"   Overlap: {len(both_sig_and_consistent)}")
else:
    # Asymmetric IQR error bars (safe non-negative)
    lower_err = (summary["mean_delta"] - summary["q25"]).clip(lower=0)
    upper_err = (summary["q75"] - summary["mean_delta"]).clip(lower=0)
    xerr = np.vstack([lower_err, upper_err])

    # Color map (RdBu_r centered at 0)
    cmap = plt.cm.RdBu_r
    norm = plt.Normalize(vmin=-abs(summary["mean_delta"]).max(), vmax=abs(summary["mean_delta"]).max())
    colors = cmap(norm(summary["mean_delta"].values))

    fig, ax = plt.subplots(figsize=(12, max(7, len(summary) * 0.4)))
    bars = ax.barh(
        summary["pair"],
        summary["mean_delta"],
        xerr=xerr,
        color=colors,
        edgecolor="none",  # remove black outlines
        linewidth=0
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Mean Δ (PT - DX)")
    ax.set_title("Consistent DX - PT Trends (celltypes_all, T/B pairs)\nError bar = IQR (Q25-Q75)")

    # Add numeric labels near zero end with auto-contrast text color
    for bar, val, col in zip(bars, summary["mean_delta"], colors):
        y = bar.get_y() + bar.get_height() / 2
        r, g, b, _ = col
        luminance = 0.299*r + 0.587*g + 0.114*b
        text_color = "black" if luminance > 0.6 else "white"

        if val >= 0:
            ax.text(0.01, y, f"{val:.2f}", va="center", ha="left", fontsize=10, color=text_color)
        else:
            ax.text(-0.01, y, f"{val:.2f}", va="center", ha="right", fontsize=10, color=text_color)

    # Clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{figdir}celltypes_all_TB_trends_barplot_signed_IQR_sigonly.pdf",
                format="pdf", bbox_inches="tight", transparent=True, dpi=600)
    plt.show()

In [ ]:
# Plot settings and color map
sample_key = "sample"
patient_key = "patient"
timepoint_key = "timepoint"
group_key = "celltype"
scalebar_len = 500  # microns

if "celltype_colors" in adata.uns and pd.api.types.is_categorical_dtype(adata.obs[group_key]):
    cats = adata.obs[group_key].cat.categories
    colors = adata.uns["celltype_colors"]
    color_map = {c: colors[i] for i, c in enumerate(cats)}
else:
    cats = sorted(pd.unique(adata.obs[group_key]))
    color_map = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}

# Build metadata and spatial bounds
meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()

mins, maxs = [], []
for s in meta[sample_key].unique():
    sub = adata[adata.obs[sample_key] == s]
    coords = sub.obsm["spatial"]
    mins.append(coords.min(axis=0))
    maxs.append(coords.max(axis=0))
mins = np.vstack(mins)
maxs = np.vstack(maxs)
global_min = mins.min(axis=0)
global_max = maxs.max(axis=0)

# Plot
for _, row in meta.iterrows():
    sample = row[sample_key]
    patient = row[patient_key]
    tp = row[timepoint_key]

    sub_nb  = adata[(adata.obs[sample_key] == sample) & (adata.obs["celltype"] == "Neuroblast")]
    sub_gdt = adata[(adata.obs[sample_key] == sample) & (adata.obs["celltypes_all"] == "γδT")]

    fig, ax = plt.subplots(figsize=(9, 9))

    if len(sub_nb) > 0:
        coords_nb = sub_nb.obsm["spatial"]
        colors_nb = [color_map.get(x, "gray") for x in sub_nb.obs["celltype"].astype(str).values]
        ax.scatter(coords_nb[:, 0], coords_nb[:, 1], s=2, c=colors_nb, linewidths=0, alpha=0.8)

    if len(sub_gdt) > 0:
        coords_gdt = sub_gdt.obsm["spatial"]
        colors_gdt = [color_map.get(x, "gray") for x in sub_gdt.obs["celltype"].astype(str).values]
        ax.scatter(coords_gdt[:, 0], coords_gdt[:, 1], s=20, c=colors_gdt, linewidths=0, alpha=0.8)

    ax.set_xlim(global_min[0], global_max[0])
    ax.set_ylim(global_min[1], global_max[1])
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(f"{patient} {tp}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    if str(tp).upper() == "DX":
        x0 = global_min[0] + 0.05 * (global_max[0] - global_min[0])
        y0 = global_max[1] - 0.05 * (global_max[1] - global_min[1])
        ax.plot([x0, x0 + scalebar_len], [y0, y0], color="black", linewidth=2)
        ax.text(x0, y0 - 0.02 * (global_max[1] - global_min[1]),
                f"{scalebar_len} µm", fontsize=8, ha="left", va="top")

    plt.tight_layout()
    plt.savefig(f"{figdir}spatial_{patient}_{tp}_NB_gdT.png",
                format="png", bbox_inches="tight", transparent=True, dpi=300)
    plt.show()
    plt.close(fig)